# The purpose of this notebook is to predict the NCAA Wrestling Tournament

In [62]:
import pandas as pd
import numpy as np
import re
import pickle
from collections import Counter
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

In [63]:
# -- read in all data
# -- 1 -- BASIC LOGREG (win_rate difference)
with open("logreg_model_OFFICIAL.pkl", "rb") as f:
    logreg_model = pickle.load(f)

# -- 2 -- DECISION TREE
with open("dt_model_tuned_OFFICIAL.pkl", "rb") as f:
    dt_model = pickle.load(f)

# -- 3 -- XGBoost w/o ranks
with open("xgboost_model_tuned_noranks_OFFICIAL.pkl", "rb") as f:
    xgb_model1 = pickle.load(f)

# -- 4 -- XGbosst w rank
with open("xgb_with_ranks_tuned_OFFICIAL.pkl", "rb") as f:
    xgb_model2 = pickle.load(f)



# -- Features Dt and XGBoost(1)
with open("features_dt_xgb1.pkl", "rb") as f:
    features_dt_xgb1 = pickle.load(f)


# -- Features XGBoost(2)
with open("xgb_w_ranks_features.pkl", "rb") as f:
    features_xgb2 = pickle.load(f)

features_xgb2 = features_xgb2.tolist()

inference_pipeline_db = pd.read_csv('inference_pipeline_db.csv')
wrestlers = pd.read_csv('wrestlers_updated.csv')
matches = pd.read_csv('matches_detailed_with_history.csv')



display(inference_pipeline_db)
display(wrestlers)
display(matches)

print(matches.columns)

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff
0,William Morrow,Bloomsburg,206,75,[157],22,3,19,13.6,1,4.5,1,0,0,2,113.41,2.68,-2.77,5.20
1,Mason Rebuck,Bloomsburg,53,75,[285],20,15,5,75.0,10,50.0,2,2,6,5,127.85,8.17,3.92,8.62
2,Noah Mulvaney,Bucknell,32,36,[165],20,17,3,85.0,7,35.0,3,3,1,10,93.00,9.22,5.33,6.26
3,Jordan Soriano,Drexel,60,44,[141],20,14,6,70.0,7,35.0,0,2,5,7,112.05,7.31,2.23,6.22
4,Jesse Mendez,Ohio State,1,2,[141],19,19,0,100.0,17,89.5,9,3,5,2,57.37,16.64,12.36,4.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,Alaa Aly,Edinboro,208,47,[184],1,0,1,0.0,0,0.0,0,0,0,0,40.00,5.00,-15.00,0.00
1502,Cole Tolley,West Virginia,114,17,[197],1,0,1,0.0,0,0.0,0,0,0,0,63.00,6.00,-15.00,0.00
1503,Winston McBride,Wyoming,182,21,[285],1,1,0,100.0,0,0.0,0,0,0,1,240.00,1.00,1.00,0.00
1504,Aiden Robertson,Air Force,222,52,[174],1,0,1,0.0,0,0.0,0,0,0,0,76.00,0.00,-5.00,0.00


,wrestler_id,name,Class,Team
0,1,Bowen Downey,SO,Northern Iowa
1,2,Julian Farber,SR,Northern Iowa
2,3,Max Brady,FR,Northern Iowa
3,4,Ethan Basile,SR,Northern Iowa
4,5,Cael Rahnavardi,SR,Northern Iowa
...,...,...,...,...
1507,38,Landen Johnson,SO,Northern Illinois
1508,168,Nathan Taylor,SR,Lehigh
1509,897,James Conway,SR,Missouri
1510,2000,James Conway,SR,Franklin & Marshall


,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,...,away_avg_point_diff,away_avg_points_scored,away_std_point_diff,home_team_id,away_team_id,home_team_rank,away_team_rank,wrestled_before,home_point_diff_rematches,home_pinned_away
0,280,141,2025-11-01,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,...,0.000000,0.000000,0.000000,24,67,72,76,0,0.0,0
1,283,197,2025-11-01,878.0,Kael Bennie,46.0,263.0,Angelo Posada,20.0,False,...,0.000000,0.000000,0.000000,70,23,36,11,0,0.0,0
2,283,285,2025-11-01,879.0,Jack Forbes,62.0,429.0,Jackson Mankowski,182.0,True,...,0.000000,0.000000,0.000000,70,23,36,11,0,0.0,0
3,283,174,2025-11-01,994.0,Tanner Lofthouse,112.0,261.0,Collin Guffey,115.0,False,...,0.000000,0.000000,0.000000,70,23,36,11,0,0.0,0
4,284,133,2025-11-01,785.0,Anthony Lucio,245.0,226.0,Blake Boarman,57.0,False,...,0.000000,0.000000,0.000000,64,20,47,32,0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5627,547,165,2026-02-23,1374.0,Brandon Jacoby,284.0,813.0,Ryan Vigil,97.0,False,...,-1.714286,5.000000,8.138679,37,66,78,71,0,0.0,0
5628,547,157,2026-02-23,1448.0,Phil Cuttino,212.0,822.0,Raymond Cmil,205.0,False,...,-9.800000,1.800000,6.300794,37,66,78,71,0,0.0,0
5629,547,174,2026-02-23,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,...,-3.909091,4.636364,7.687061,37,66,78,71,0,0.0,0
5630,547,184,2026-02-23,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,...,-6.300000,6.800000,9.499123,37,66,78,71,0,0.0,0


Index(['dual_id', 'weight_class', 'event_date', 'home_wrestler_id',
       'home_name', 'home_rank', 'away_wrestler_id', 'away_name', 'away_rank',
       'home_win', 'win_type', 'Result', 'home_class', 'home_team_name',
       'away_class', 'away_team_name', 'home_matches_wrestled',
       'home_win_rate', 'home_loss_rate', 'home_bonus_rate', 'home_pin_count',
       'home_avg_opponent_rank', 'home_avg_point_diff',
       'home_avg_points_scored', 'home_std_point_diff',
       'away_matches_wrestled', 'away_win_rate', 'away_loss_rate',
       'away_bonus_rate', 'away_pin_count', 'away_avg_opponent_rank',
       'away_avg_point_diff', 'away_avg_points_scored', 'away_std_point_diff',
       'home_team_id', 'away_team_id', 'home_team_rank', 'away_team_rank',
       'wrestled_before', 'home_point_diff_rematches', 'home_pinned_away'],
      dtype='object')


In [64]:
matches.query('home_name == "Drake Ayala" or away_name == "Drake Ayala"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,...,away_avg_point_diff,away_avg_points_scored,away_std_point_diff,home_team_id,away_team_id,home_team_rank,away_team_rank,wrestled_before,home_point_diff_rematches,home_pinned_away
93,275,133,2025-11-06,196.0,Drake Ayala,6.0,1005.0,Trayce Eckman,74.0,True,...,0.000000,0.000000,0.000000,17,77,2,54,0,0.0,0
401,219,133,2025-11-15,196.0,Drake Ayala,6.0,928.0,Kade Moore,29.0,True,...,0.000000,0.000000,0.000000,17,73,2,15,0,0.0,0
521,206,133,2025-11-15,196.0,Drake Ayala,6.0,140.0,Lucas Byrd,1.0,False,...,11.000000,14.333333,6.928203,17,13,2,10,0,0.0,0
826,188,133,2025-11-16,443.0,Ben Davino,3.0,196.0,Drake Ayala,6.0,True,...,5.000000,10.500000,14.142136,41,17,4,2,0,0.0,0
834,187,133,2025-11-16,196.0,Drake Ayala,6.0,591.0,Ronnie Ramirez,28.0,True,...,4.000000,6.500000,2.828427,17,52,2,5,0,0.0,0
1006,178,133,2025-11-21,196.0,Drake Ayala,6.0,295.0,Evan Tallmadge,50.0,True,...,7.666667,9.666667,6.429101,17,27,2,19,0,0.0,0
1185,162,133,2025-11-30,600.0,Evan Frost,5.0,196.0,Drake Ayala,6.0,True,...,3.200000,9.000000,9.011104,39,17,6,2,0,0.0,0
2542,29,133,2026-01-09,196.0,Drake Ayala,6.0,206.0,Zan Fugitt,10.0,False,...,5.142857,8.142857,7.883074,17,18,2,31,0,0.0,0
3158,324,133,2026-01-16,196.0,Drake Ayala,9.0,120.0,Marcus Blaze,4.0,False,...,12.000000,15.000000,3.559026,17,11,6,1,0,0.0,0
3455,373,133,2026-01-23,216.0,Jacob Van Dee,11.0,196.0,Drake Ayala,8.0,False,...,0.875000,7.125000,7.661359,19,17,4,5,0,0.0,0


In [65]:
matches.win_type.unique()

array(['LDEC', 'WDEC', 'WTB', 'LSV', 'WMD', 'LMD', 'LTF', 'WTF', 'LTB',
       'WSV', 'WFALL', 'LFALL'], dtype=object)

In [66]:
print(f"Features DT/XGB1: {len(features_dt_xgb1)} features")
print(f"Features XGB2: {len(features_xgb2)} features")

Features DT/XGB1: 33 features
Features XGB2: 34 features


# Test Predictions first

In [67]:
print("\n🔍 Merging class data from wrestlers table...")

# Merge class information
inference_db_with_class = inference_pipeline_db.merge(
    wrestlers[['name', 'Class']],
    left_on='wrestler_name',
    right_on='name',
    how='left'
)

# Drop the duplicate name column
inference_db_with_class = inference_db_with_class.drop('name', axis=1)

# Check for any missing classes
missing_class = inference_db_with_class[inference_db_with_class['Class'].isna()]
print(f"Wrestlers with missing class: {len(missing_class)}")

if len(missing_class) > 0:
    print("\nSample of missing classes:")
    print(missing_class[['wrestler_name', 'team_name']].head(10))

print("\n✅ Inference DB now has class information")
print(f"Columns: {inference_db_with_class.columns.tolist()}")

# ============================================
# FUNCTION TO GET MATCH HISTORY
# ============================================

def get_match_history(w1_name, w2_name, matches_df):
    """
    Get complete match history between two wrestlers.
    Returns historical features for prediction.
    """

    # Find all matches between these two wrestlers
    mask = (
        ((matches_df['home_name'] == w1_name) & (matches_df['away_name'] == w2_name)) |
        ((matches_df['home_name'] == w2_name) & (matches_df['away_name'] == w1_name))
    )

    history = matches_df[mask].copy()

    if len(history) == 0:
        return {
            'wrestled_before': 0,
            'home_point_diff_rematches': 0,
            'home_pinned_away': 0,
            'avg_point_diff_w1': 0,
            'pins_by_w1': 0,
            'pins_by_w2': 0
        }

    # Calculate head-to-head stats
    w1_wins = 0
    w2_wins = 0
    total_point_diff_w1 = 0
    point_diff_count = 0
    pins_by_w1 = 0
    pins_by_w2 = 0

    for idx, row in history.iterrows():
        # Determine winner
        if row['home_name'] == w1_name:
            winner = w1_name if row['home_win'] else w2_name
        else:
            winner = w2_name if row['home_win'] else w1_name

        # Calculate point differential from w1's perspective
        if 'FALL' not in str(row['win_type']):
            try:
                parts = str(row['Result']).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    score1, score2 = scores[0], scores[1]

                    if row['home_name'] == w1_name:
                        if row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1
                    else:
                        if not row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1

                    total_point_diff_w1 += point_diff
                    point_diff_count += 1
            except:
                pass

        # Check for pins
        if 'FALL' in str(row['win_type']):
            if winner == w1_name:
                pins_by_w1 += 1
            else:
                pins_by_w2 += 1

        # Count wins
        if winner == w1_name:
            w1_wins += 1
        else:
            w2_wins += 1

    # Calculate averages
    avg_point_diff_w1 = total_point_diff_w1 / point_diff_count if point_diff_count > 0 else 0

    return {
        'wrestled_before': 1 if len(history) > 0 else 0,
        'home_point_diff_rematches': round(avg_point_diff_w1, 3),
        'home_pinned_away': 1 if pins_by_w1 > 0 else 0,
        'avg_point_diff_w1': round(avg_point_diff_w1, 3),
        'pins_by_w1': pins_by_w1,
        'pins_by_w2': pins_by_w2,
        'w1_wins': w1_wins,
        'w2_wins': w2_wins
    }

# ============================================
# VERIFY WRESTLERS ARE IN DATABASE
# ============================================

test_wrestlers = ['Drake Ayala', 'Ben Davino', 'Nick Feldman', 'AJ Ferrari']

print("\n🔍 Verifying wrestlers in inference database with class:")
for wrestler in test_wrestlers:
    data = inference_db_with_class[inference_db_with_class['wrestler_name'] == wrestler]
    if len(data) > 0:
        row = data.iloc[0]
        print(f"✅ {wrestler}:")
        print(f"   Team: {row['team_name']}, Rank: {row['rank']}, Win Rate: {row['win_rate']}%")
        print(f"   Class: {row['Class']}, Weight Class(es): {row['weight_classes']}")
    else:
        print(f"❌ {wrestler} NOT FOUND in database")

# ============================================
# PREDICTION FUNCTION FOR SINGLE MATCHUP
# ============================================

def predict_matchup(w1_name, w2_name, weight_class, inference_db, matches_df,
                    logreg_model, dt_model, xgb_model1, xgb_model2,
                    features_dt_xgb1, features_xgb2, verbose=True):
    """
    Predict outcome between two wrestlers.
    Higher rank = better (lower number) is set as home.
    """

    # Get wrestler data
    w1_data = inference_db[inference_db['wrestler_name'] == w1_name].iloc[0].copy()
    w2_data = inference_db[inference_db['wrestler_name'] == w2_name].iloc[0].copy()

    # Set home/away based on rank (lower rank number = better = home)
    if w1_data['rank'] < w2_data['rank']:
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
    else:
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name

    # Get match history
    history = get_match_history(w1_name, w2_name, matches_df)

    if verbose:
        print(f"\n📌 Matchup: {w1_name} vs {w2_name}")
        print(f"   Home: {home_name} (Rank {home['rank']})")
        print(f"   Away: {away_name} (Rank {away['rank']})")
        print(f"   Weight Class: {weight_class} lbs")
        print(f"   Home Class: {home['Class']}, Away Class: {away['Class']}")
        if history['wrestled_before']:
            print(f"   Prior meetings: {history['w1_wins']}-{history['w2_wins']} (avg point diff: {history['avg_point_diff_w1']})")

    # Calculate differential features
    features = {}

    # Basic differentials
    features['matches_wrestled_diff'] = home['total_matches'] - away['total_matches']
    features['win_rate_diff'] = round(home['win_rate']/100 - away['win_rate']/100, 4)
    features['bonus_rate_diff'] = round(home['bonus_rate']/100 - away['bonus_rate']/100, 4)
    features['pin_count_diff'] = home['fall_wins'] - away['fall_wins']
    features['avg_point_diff_diff'] = round(home['avg_point_diff'] - away['avg_point_diff'], 4)
    features['avg_points_scored_diff'] = round(home['avg_points_scored'] - away['avg_points_scored'], 4)
    features['std_point_diff_diff'] = round(home['std_point_diff'] - away['std_point_diff'], 4)
    features['team_rank_diff'] = home['team_rank'] - away['team_rank']

    # Historical features - oriented to home wrestler
    if home_name == w1_name:
        # Home is w1
        features['wrestled_before'] = history['wrestled_before']
        features['home_point_diff_rematches'] = history['home_point_diff_rematches']
        features['home_pinned_away'] = history['home_pinned_away']
    else:
        # Home is w2, flip the historical features
        features['wrestled_before'] = history['wrestled_before']
        features['home_point_diff_rematches'] = -history['avg_point_diff_w1']
        features['home_pinned_away'] = 1 if history['pins_by_w2'] > 0 else 0

    # Weight class dummies
    weight_classes = [133, 141, 149, 157, 165, 174, 184, 197, 285]
    for wc in weight_classes:
        features[f'weight_class_{wc}'] = 1 if weight_class == wc else 0

    # Class dummies
    all_classes = ['JR', 'RSFR', 'RSJR', 'RSSO', 'RSSR', 'SO', 'SR']
    for cls in all_classes:
        features[f'home_class_{cls}'] = 1 if home['Class'] == cls else 0
    for cls in all_classes:
        features[f'away_class_{cls}'] = 1 if away['Class'] == cls else 0

    if verbose:
        print("\n📊 Feature DataFrame being passed to models:")

    # Create DataFrames for models
    df_dt = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb1}])
    df_xgb2 = pd.DataFrame([{f: features.get(f, 0) for f in features_xgb2}])

    if verbose:
        print("\n🔴 DT/XGB1 Features:")
        dt_display = df_dt[features_dt_xgb1].T.round(4)
        dt_display.columns = ['Value']
        display(dt_display)

        print("\n🔵 XGB2 Features:")
        xgb2_display = df_xgb2[features_xgb2].T.round(4)
        xgb2_display.columns = ['Value']
        display(xgb2_display)

    # Make predictions
    # LOGREG (uses only win_rate_diff)
    logreg_input = [[features['win_rate_diff']]]
    logreg_pred = logreg_model.predict(logreg_input)[0]
    logreg_prob_home = logreg_model.predict_proba(logreg_input)[0][1]

    # DT
    dt_pred = dt_model.predict(df_dt[features_dt_xgb1])[0]
    dt_prob_home = dt_model.predict_proba(df_dt[features_dt_xgb1])[0][1]

    # XGB1
    xgb1_pred = xgb_model1.predict(df_dt[features_dt_xgb1])[0]
    xgb1_prob_home = xgb_model1.predict_proba(df_dt[features_dt_xgb1])[0][1]

    # XGB2
    xgb2_pred = xgb_model2.predict(df_xgb2[features_xgb2])[0]
    xgb2_prob_home = xgb_model2.predict_proba(df_xgb2[features_xgb2])[0][1]

    # Determine winners and probabilities from winner's perspective
    results = {}

    # LOGREG
    if logreg_pred == 1:
        results['logreg_winner'] = home_name
        results['logreg_prob'] = logreg_prob_home
    else:
        results['logreg_winner'] = away_name
        results['logreg_prob'] = 1 - logreg_prob_home

    # DT
    if dt_pred == 1:
        results['dt_winner'] = home_name
        results['dt_prob'] = dt_prob_home
    else:
        results['dt_winner'] = away_name
        results['dt_prob'] = 1 - dt_prob_home

    # XGB1
    if xgb1_pred == 1:
        results['xgb1_winner'] = home_name
        results['xgb1_prob'] = xgb1_prob_home
    else:
        results['xgb1_winner'] = away_name
        results['xgb1_prob'] = 1 - xgb1_prob_home

    # XGB2
    if xgb2_pred == 1:
        results['xgb2_winner'] = home_name
        results['xgb2_prob'] = xgb2_prob_home
    else:
        results['xgb2_winner'] = away_name
        results['xgb2_prob'] = 1 - xgb2_prob_home

    return results, home_name, away_name, features

# ============================================
# TEST CASE 1: Drake Ayala vs Ben Davino (133 lbs)
# ============================================

print("\n" + "="*100)
print("🏆 TEST CASE 1: Drake Ayala vs Ben Davino (133 lbs)")
print("="*100)

results1, home1, away1, features1 = predict_matchup(
    "Drake Ayala", "Ben Davino", 133, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2, verbose=True
)

print("\n📊 PREDICTION RESULTS:")
print("-" * 60)
print(f"{'Model':<25} {'Predicted Winner':<25} {'Confidence':<15}")
print("-" * 60)
print(f"{'Logistic Regression':<25} {results1['logreg_winner']:<25} {results1['logreg_prob']*100:.1f}%")
print(f"{'Decision Tree':<25} {results1['dt_winner']:<25} {results1['dt_prob']*100:.1f}%")
print(f"{'XGBoost (no ranks)':<25} {results1['xgb1_winner']:<25} {results1['xgb1_prob']*100:.1f}%")
print(f"{'XGBoost (with ranks)':<25} {results1['xgb2_winner']:<25} {results1['xgb2_prob']*100:.1f}%")

# ============================================
# TEST CASE 2: Nick Feldman vs AJ Ferrari (285 lbs)
# ============================================

print("\n" + "="*100)
print("🏆 TEST CASE 2: Nick Feldman vs AJ Ferrari (285 lbs)")
print("="*100)

results2, home2, away2, features2 = predict_matchup(
    "Nick Feldman", "AJ Ferrari", 285, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2, verbose=True
)

print("\n📊 PREDICTION RESULTS:")
print("-" * 60)
print(f"{'Model':<25} {'Predicted Winner':<25} {'Confidence':<15}")
print("-" * 60)
print(f"{'Logistic Regression':<25} {results2['logreg_winner']:<25} {results2['logreg_prob']*100:.1f}%")
print(f"{'Decision Tree':<25} {results2['dt_winner']:<25} {results2['dt_prob']*100:.1f}%")
print(f"{'XGBoost (no ranks)':<25} {results2['xgb1_winner']:<25} {results2['xgb1_prob']*100:.1f}%")
print(f"{'XGBoost (with ranks)':<25} {results2['xgb2_winner']:<25} {results2['xgb2_prob']*100:.1f}%")

# ============================================
# SUMMARY TABLE
# ============================================

print("\n" + "="*100)
print("📊 SUMMARY TABLE")
print("="*100)

summary_data = [
    {
        'Matchup': f"{home1} vs {away1}",
        'Weight': '133 lbs',
        'LOGREG Winner': results1['logreg_winner'],
        'LOGREG Conf': f"{results1['logreg_prob']*100:.1f}%",
        'DT Winner': results1['dt_winner'],
        'DT Conf': f"{results1['dt_prob']*100:.1f}%",
        'XGB1 Winner': results1['xgb1_winner'],
        'XGB1 Conf': f"{results1['xgb1_prob']*100:.1f}%",
        'XGB2 Winner': results1['xgb2_winner'],
        'XGB2 Conf': f"{results1['xgb2_prob']*100:.1f}%",
    },
    {
        'Matchup': f"{home2} vs {away2}",
        'Weight': '285 lbs',
        'LOGREG Winner': results2['logreg_winner'],
        'LOGREG Conf': f"{results2['logreg_prob']*100:.1f}%",
        'DT Winner': results2['dt_winner'],
        'DT Conf': f"{results2['dt_prob']*100:.1f}%",
        'XGB1 Winner': results2['xgb1_winner'],
        'XGB1 Conf': f"{results2['xgb1_prob']*100:.1f}%",
        'XGB2 Winner': results2['xgb2_winner'],
        'XGB2 Conf': f"{results2['xgb2_prob']*100:.1f}%",
    }
]

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print("\n✅ Test cases complete!")


🔍 Merging class data from wrestlers table...
Wrestlers with missing class: 0

✅ Inference DB now has class information
Columns: ['wrestler_name', 'team_name', 'rank', 'team_rank', 'weight_classes', 'total_matches', 'total_wins', 'total_losses', 'win_rate', 'bonus_wins', 'bonus_rate', 'tf_wins', 'md_wins', 'fall_wins', 'dec_wins', 'avg_opponent_rank', 'avg_points_scored', 'avg_point_diff', 'std_point_diff', 'Class']

🔍 Verifying wrestlers in inference database with class:
✅ Drake Ayala:
   Team: Iowa, Rank: 10, Win Rate: 56.2%
   Class: SR, Weight Class(es): [133]
✅ Ben Davino:
   Team: Ohio State, Rank: 4, Win Rate: 94.7%
   Class: RSFR, Weight Class(es): [133]
✅ Nick Feldman:
   Team: Ohio State, Rank: 5, Win Rate: 76.5%
   Class: JR, Weight Class(es): [285]
✅ AJ Ferrari:
   Team: Nebraska, Rank: 3, Win Rate: 86.7%
   Class: JR, Weight Class(es): [285]

🏆 TEST CASE 1: Drake Ayala vs Ben Davino (133 lbs)

📌 Matchup: Drake Ayala vs Ben Davino
   Home: Ben Davino (Rank 4)
   Away: Drake

,Value
wrestled_before,1.000
home_point_diff_rematches,4.000
home_pinned_away,0.000
weight_class_133,1.000
weight_class_141,0.000
weight_class_149,0.000
weight_class_157,0.000
weight_class_165,0.000
weight_class_174,0.000
weight_class_184,0.000



🔵 XGB2 Features:


,Value
wrestled_before,1.000
home_point_diff_rematches,4.000
home_pinned_away,0.000
matches_wrestled_diff,3.000
win_rate_diff,0.385
bonus_rate_diff,0.194
pin_count_diff,0.000
avg_point_diff_diff,6.160
avg_points_scored_diff,2.770
std_point_diff_diff,-4.170



📊 PREDICTION RESULTS:
------------------------------------------------------------
Model                     Predicted Winner          Confidence     
------------------------------------------------------------
Logistic Regression       Ben Davino                70.0%
Decision Tree             Ben Davino                74.6%
XGBoost (no ranks)        Ben Davino                78.0%
XGBoost (with ranks)      Ben Davino                72.7%

🏆 TEST CASE 2: Nick Feldman vs AJ Ferrari (285 lbs)

📌 Matchup: Nick Feldman vs AJ Ferrari
   Home: AJ Ferrari (Rank 3)
   Away: Nick Feldman (Rank 5)
   Weight Class: 285 lbs
   Home Class: JR, Away Class: JR
   Prior meetings: 2-0 (avg point diff: 2.0)

📊 Feature DataFrame being passed to models:

🔴 DT/XGB1 Features:


...


🔵 XGB2 Features:


,Value
wrestled_before,1.000
home_point_diff_rematches,-2.000
home_pinned_away,0.000
matches_wrestled_diff,-2.000
win_rate_diff,0.102
bonus_rate_diff,0.055
pin_count_diff,0.000
avg_point_diff_diff,0.550
avg_points_scored_diff,-0.460
std_point_diff_diff,-1.500



📊 PREDICTION RESULTS:
------------------------------------------------------------
Model                     Predicted Winner          Confidence     
------------------------------------------------------------
Logistic Regression       AJ Ferrari                55.8%
Decision Tree             AJ Ferrari                61.1%
XGBoost (no ranks)        AJ Ferrari                58.4%
XGBoost (with ranks)      AJ Ferrari                50.8%

📊 SUMMARY TABLE


,Matchup,Weight,LOGREG Winner,LOGREG Conf,DT Winner,DT Conf,XGB1 Winner,XGB1 Conf,XGB2 Winner,XGB2 Conf
0,Ben Davino vs Drake Ayala,133 lbs,Ben Davino,70.0%,Ben Davino,74.6%,Ben Davino,78.0%,Ben Davino,72.7%
1,AJ Ferrari vs Nick Feldman,285 lbs,AJ Ferrari,55.8%,AJ Ferrari,61.1%,AJ Ferrari,58.4%,AJ Ferrari,50.8%



✅ Test cases complete!


# Create wrestler list

In [68]:
# 125 lbs (33 wrestlers)
wrestlers_125 = [
    'Jace Schafer', 'Mack Mauger', 'Luke Lilledahl', 'Jett Strickenberger',
    'Ezekiel Witt', 'Maximo Renteria', 'Ayden Smith', 'Kael Lauridsen',
    'Dean Peterson', 'Troy Spratley', 'Andrew Binni', 'Conrad Hendriksen',
    'Vincent Robinson', 'Stevo Poulin', 'Diego Sotelo', 'Tyler Chappell',
    'Sheldon Seymour', 'Nic Bouzakis', 'Sulayman Bah', 'Kysen Terukina',
    'Jacob Moran', 'Tyler Klinsky', 'Davis Motyka', 'Brady Roark',
    'Jore Volk', 'Nico Provo', 'Cooper Flynn', 'Nicolar Rivera',
    'Marc-Anthony McGowan', 'Koda Holeman', 'Spencer Moore', 'Desmond Pleasant',
    'Eddie Ventresca'
]

# 133 lbs (33 wrestlers)
wrestlers_133 = [
    'Carter Schmidt', 'Andrew Austin', 'Jax Forrest', 'TK Davis',
    'Zan Fugitt', 'Dominick Serrano', 'Blake Boarman', 'Will Betancourt',
    'Markel Baker', 'Kyler Larkin', 'Garrett Grice', 'Sean Spidle',
    'Evan Mougalian', 'Jacob Van Dee', 'Julian Farber', 'Luke Willochell',
    'Aaron Seidel', 'Marcus Blaze', 'Gabe Whisenhunt', 'Gage Walker',
    'Ethan Berginc', 'Tyler Ferrera', 'Zach Redding', 'Marcel Lopez',
    'Drake Ayala', 'Lucas Byrd', 'Dylan Shawver', 'Braxton Brown',
    'Maximillian Leete', 'Tyler Knox', 'Gunner Andrick', 'Gable Strickland',
    'Ben Davino'
]

# 141 lbs (33 wrestlers)
wrestlers_141 = [
    'Aldo Hernandez', 'Matthew Martino', 'Jesse Mendez', 'Caedyn Ricciardi',
    'Ryan Jack', 'Joey Olivieri', 'Nash Singleton', 'Tom Crook',
    'Vance Vombaur', 'Luke Stanich', 'Pierson Manville', 'Tyler Wells',
    'Luke Simcox', 'Wyatt Henson', 'Julian Tagg', 'Jordan Titus',
    'Anthony Echemendia', 'Brock Hardy', 'Dario Lemus', 'Haiden Drury',
    'Braeden Davis', 'CJ Composto', 'Lorenzo Frezza', 'Gable Porter',
    'Vince Cornella', 'Nasir Bailey', 'Braden Basile', 'Dylan Chappell',
    'Jack Consiglio', 'Eli Griffin', 'Carter Nogle', 'Billy Dekraker',
    'Sergio Vega'
]

# 149 lbs (33 wrestlers)
wrestlers_149 = [
    'Austin McBurney', 'Clayton Jones', 'Shayne Van Ness', 'Lucas Kapusta',
    'Jacob Frost', 'David Evans', 'Andrew Clark', 'Michael Gioffre',
    'Casey Swiderski', 'Koy Buesgens', 'Kade Brown', 'Gabe Willochell',
    'Carter Young', 'Joseph Zargo', 'Chance Lamer', 'Kaden Cassidy',
    'Collin Gaj', 'Cross Wasilewski', 'Dylan Layton', 'Brock Herman',
    'Caleb Rathjen', 'Lachlan McNeil', 'Eligh Rivera', 'Andre Gonzales',
    'Caleb Tyus', 'Ethan Stiles', 'Anderson Heap', 'Maxwell Petersen',
    'Aden Valencia', 'Ryder Block', 'Eugene Harney', 'Ryan Michaels',
    'Jaxon Joy'
]

# 157 lbs (33 wrestlers)
wrestlers_157 = [
    'Yannis Charles', 'Jeb Prechtel', 'PJ Duke', 'Luke Mechler',
    'Cael Swensen', 'Daniel Cardenas', 'Jaivon Jones', 'Mason Shrader',
    'Brandon Cannon', 'Landon Robideau', 'Gavin Drexler', 'Charlie Millard',
    'Vince Zerban', 'Derek Raike', 'Jimmy Harrington', 'Bryce Lowery',
    'Kaleb Larkin', 'Meyer Shapiro', 'Laird Root', 'Kai Owen',
    'Ethen Miller', 'Ty Watters', 'Colton Washleski', 'Dylan Evans',
    'Jude Swisher', 'Kannon Webster', 'Jonathan Ley', 'Kaleb Burgess',
    'Logan Rozynski', 'Cam Catrabone', 'DJ McGee', 'Garrett McChesney',
    'Antrell Taylor'
]

# 165 lbs (33 wrestlers)
wrestlers_165 = [
    'Ryan Vigil', 'Cody Walsh', 'Mitchell Mesenbrink', 'Braeden Scoles',
    'Paddy Gallagher', 'Bryce Hepner', 'Sean Seefeldt', 'Mac Church',
    'Matt Bianchi', 'Ladarion Lockett', 'Cody Goebel', 'Brock Woodcock',
    'Cesar Alvan', 'Andrew Sparks', 'Ty Whalen', 'Ryan Burgos',
    'Nicco Ruiz', 'Michael Caliendo', 'Thomas Snipes', 'Noah Mulvaney',
    'Andrew Barbosa', 'Ryder Downey', 'Matthew Olguin', 'EJ Parco',
    'LJ Araujo', 'Max Brignola', 'Tyler Lillard', 'Chris Earnest',
    'Will Denny', 'Connor Euton', 'Gunner Filipowicz', 'Jared Keslar',
    'Joey Blaze'
]

# 174 lbs (33 wrestlers)
wrestlers_174 = [
    "Grant O'Dell", 'Lucas Condon', 'Levi Haines', 'Jared Simma',
    'Nick Fine', 'Beau Mantanona', 'Garrett Thompson', 'Sergio Desiante',
    'Alex Facundo', 'Patrick Kennedy', 'Holden Garcia', 'Lenny Pinto',
    'Carter Schubert', 'Carter Baer', 'Daschle Lamer', 'Avery Bassett',
    'Carson Kharchla', 'Christopher Minto', 'Riley Davis', 'Logan Messer',
    'Moses Espinoza-Owens', 'MJ Gaitan', 'Brody Baumann', 'Collin Carrigan',
    'Matthew Singleton', 'Cameron Steed', 'Derek Gilcher', 'Luca Augustine',
    'Myles Takats', 'Danny Wask', 'Colin Kelly', 'Cael Valencia',
    'Simon Ruiz'
]

# 184 lbs (33 wrestlers)
wrestlers_184 = [
    'Sam Goin', 'Caleb Uhlenhopp', 'Rocco Welsh', 'Ian Bush',
    'Rylan Rogers', 'Chris Moore', 'Joe Curtis', 'Malachi Duvall',
    'Silas Allred', 'Brock Mantanona', 'Abe Wojcikiewicz', 'Tomas Brooker',
    'Dylan Fishback', 'Isaac Dean', 'Brian Soldano', 'Nick Fox',
    'James Conway', 'Max McEnelly', 'Tyler Bienus', 'Jared McGill',
    'Jaden Bullock', 'Shane Cartagena-Walsh', 'Zack Ryder', 'Aidan Brenot',
    'Eddie Neitenbach', 'Angelo Ferrari', 'Chase Kranitz', 'Ceasar Garza',
    'Caleb Campos', 'Sal Perrine', 'Jake Dailey', 'Mahonri Rushton',
    'Aeoden Sinclair'
]

# 197 lbs (33 wrestlers)
wrestlers_197 = [
    'Karson Tompkins', 'Blake Schaffer', 'Josh Barr', 'Dillon Bechtold',
    'Branson John', 'Angelo Posada', 'Brock Zurawski', 'Evan Bates',
    'Deanthony Parker', 'Joey Novak', 'Kael Wisler', 'Rune Lawrence',
    'Luke Geog', 'Bennett Berge', 'Wyatt Ingham', 'Colton Hawks',
    'Sonny Sasso', 'Stephen Little', 'Kade Rule', 'Zayne Lehman',
    'Gabe Sollars', 'Camden McDanel', 'Devin Wasley', 'Gabe Arnold',
    'Justin Rademacher', 'Cody Merrill', 'Ben Vanadia', 'Mikey Squires',
    'Mac Stout', 'Remy Cotton', 'Andrew Reall', 'Kael Bennie',
    'Rocky Elam'
]

# 285 lbs (33 wrestlers)
wrestlers_285 = [
    'Mason Rebuck', 'Emmanuel Ulrich', 'Yonger Bastida', 'Vincent Mueller',
    'Jim Mullen', 'Cole Mirasola', 'Connor Barket', 'Alex Semenenko',
    'Ben Kueter', 'Nick Feldman', 'Jarrett Stoner', 'Juan Mora',
    'Braxton Amos', 'Spencer Lanogsa', 'Dayton Pitzer', 'Luke Rasmussen',
    'AJ Ferrari', 'Taye Ghadiali', 'Jack Forbes', 'Nate Schon',
    'Koy Hopke', 'Devon Dawson', 'Trevor Tinker', 'Hunter Catka',
    'Nathan Taylor', 'Konner Doucet', 'Luke Luffman', 'Stephan Monchery',
    'David Szuba', 'Brady Colbert', 'Christian Carroll', 'Brenan Morgan',
    'Isaac Trumble'
]

# Group all weight classes
all_wrestlers_by_weight = {
    125: wrestlers_125,
    133: wrestlers_133,
    141: wrestlers_141,
    149: wrestlers_149,
    157: wrestlers_157,
    165: wrestlers_165,
    174: wrestlers_174,
    184: wrestlers_184,
    197: wrestlers_197,
    285: wrestlers_285
}

# Verify each weight class has 33 wrestlers
print("\n" + "="*100)
print("📋 VERIFYING NCAA QUALIFIER COUNTS")
print("="*100)

for weight, wrestler_list in all_wrestlers_by_weight.items():
    count = len(wrestler_list)
    status = "✅" if count == 33 else "❌"
    print(f"{weight}lbs: {count} wrestlers {status}")

    # Check if all wrestlers are in inference_db
    missing = []
    for w in wrestler_list:
        if len(inference_pipeline_db[inference_pipeline_db['wrestler_name'] == w]) == 0:
            missing.append(w)

    if missing:
        print(f"   ⚠️ Missing {len(missing)} wrestlers: {', '.join(missing[:5])}{'...' if len(missing) > 5 else ''}")



📋 VERIFYING NCAA QUALIFIER COUNTS
125lbs: 33 wrestlers ✅
133lbs: 33 wrestlers ✅
141lbs: 33 wrestlers ✅
149lbs: 33 wrestlers ✅
157lbs: 33 wrestlers ✅
165lbs: 33 wrestlers ✅
174lbs: 33 wrestlers ✅
184lbs: 33 wrestlers ✅
197lbs: 33 wrestlers ✅
285lbs: 33 wrestlers ✅


# Make Predictions by weight class

In [69]:
# ============================================
# NCAA PREDICTION FUNCTIONS
# ============================================

def get_wrestler_match_history(wrestler1_name, wrestler2_name, matches_df):
    """
    Get complete match history between two wrestlers from matches_reference.
    Returns DataFrame of all matches and calculated historical features.
    """

    # Find all matches between these two wrestlers
    mask = (
        ((matches_df['home_name'] == wrestler1_name) & (matches_df['away_name'] == wrestler2_name)) |
        ((matches_df['home_name'] == wrestler2_name) & (matches_df['away_name'] == wrestler1_name))
    )

    history = matches_df[mask].copy().sort_values('event_date')

    if len(history) == 0:
        return {
            'wrestled_before': 0,
            'home_point_diff_rematches': 0,
            'home_pinned_away': 0,
            'total_matches': 0,
            'w1_wins': 0,
            'w2_wins': 0,
            'avg_point_diff_w1': 0,
            'pins_by_w1': 0,
            'pins_by_w2': 0,
            'last_match_date': None,
            'last_winner': None
        }

    # Calculate head-to-head stats
    w1_name = wrestler1_name
    w2_name = wrestler2_name

    w1_wins = 0
    w2_wins = 0
    total_point_diff_w1 = 0
    point_diff_count = 0
    pins_by_w1 = 0
    pins_by_w2 = 0

    # Store last winner
    last_winner = None

    for idx, row in history.iterrows():
        # Determine winner
        if row['home_name'] == w1_name:
            winner = w1_name if row['home_win'] else w2_name
        else:
            winner = w2_name if row['home_win'] else w1_name

        # Update last winner
        last_winner = winner

        # Calculate point differential (from w1's perspective)
        if 'FALL' not in str(row['win_type']):
            try:
                parts = str(row['Result']).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    score1, score2 = scores[0], scores[1]

                    # Determine scores from w1's perspective
                    if row['home_name'] == w1_name:
                        if row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1
                    else:
                        if not row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1

                    total_point_diff_w1 += point_diff
                    point_diff_count += 1
            except:
                pass

        # Check for pins
        if 'FALL' in str(row['win_type']):
            if winner == w1_name:
                pins_by_w1 += 1
            else:
                pins_by_w2 += 1

        # Count wins
        if winner == w1_name:
            w1_wins += 1
        else:
            w2_wins += 1

    # Calculate averages
    avg_point_diff_w1 = total_point_diff_w1 / point_diff_count if point_diff_count > 0 else 0

    return {
        'wrestled_before': 1 if len(history) > 0 else 0,
        'home_point_diff_rematches': round(avg_point_diff_w1, 3),
        'home_pinned_away': 1 if pins_by_w1 > 0 else 0,
        'total_matches': len(history),
        'w1_wins': w1_wins,
        'w2_wins': w2_wins,
        'avg_point_diff_w1': round(avg_point_diff_w1, 3),
        'pins_by_w1': pins_by_w1,
        'pins_by_w2': pins_by_w2,
        'last_match_date': history.iloc[-1]['event_date'] if len(history) > 0 else None,
        'last_winner': last_winner
    }


# ============================================
# FUNCTION TO PREPARE FEATURES FOR A MATCHUP
# ============================================

def prepare_matchup_features(w1_name, w2_name, historical_features, inference_db, weight_class=125):
    """
    Prepare all features needed for models given two wrestlers.
    Home is determined by better rank (lower number).
    """

    # Get wrestler data from inference pipeline
    w1_data = inference_db[inference_db['wrestler_name'] == w1_name].iloc[0].copy()
    w2_data = inference_db[inference_db['wrestler_name'] == w2_name].iloc[0].copy()

    # Determine home/away based on rank (lower rank number = better = home)
    if w1_data['rank'] < w2_data['rank']:
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
    else:
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name

    # Calculate differential features
    features = {}

    # Basic differentials
    features['matches_wrestled_diff'] = home['total_matches'] - away['total_matches']
    features['win_rate_diff'] = round(home['win_rate']/100 - away['win_rate']/100, 4)
    features['bonus_rate_diff'] = round(home['bonus_rate']/100 - away['bonus_rate']/100, 4)
    features['pin_count_diff'] = home['fall_wins'] - away['fall_wins']
    features['avg_point_diff_diff'] = round(home['avg_point_diff'] - away['avg_point_diff'], 4)
    features['avg_points_scored_diff'] = round(home['avg_points_scored'] - away['avg_points_scored'], 4)
    features['std_point_diff_diff'] = round(home['std_point_diff'] - away['std_point_diff'], 4)
    features['team_rank_diff'] = home['team_rank'] - away['team_rank']

    # Historical features - oriented to home wrestler
    if home_name == w1_name:
        # Home is w1
        features['wrestled_before'] = historical_features['wrestled_before']
        features['home_point_diff_rematches'] = historical_features['home_point_diff_rematches']
        features['home_pinned_away'] = historical_features['home_pinned_away']
    else:
        # Home is w2, flip historical features
        features['wrestled_before'] = historical_features['wrestled_before']
        features['home_point_diff_rematches'] = -historical_features['avg_point_diff_w1']
        features['home_pinned_away'] = 1 if historical_features['pins_by_w2'] > 0 else 0

    # Weight class dummies
    weight_classes = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]
    for wc in weight_classes:
        features[f'weight_class_{wc}'] = 1 if weight_class == wc else 0

    # Class dummies
    all_classes = ['JR', 'RSFR', 'RSJR', 'RSSO', 'RSSR', 'SO', 'SR']

    for cls in all_classes:
        features[f'home_class_{cls}'] = 1 if home['Class'] == cls else 0

    for cls in all_classes:
        features[f'away_class_{cls}'] = 1 if away['Class'] == cls else 0

    return features, home_name, away_name, home, away


# ============================================
# FUNCTION TO MAKE PREDICTIONS FOR ONE MATCHUP
# ============================================

def predict_single_matchup(w1_name, w2_name, features, home_name, away_name, weight_class):
    """
    Make predictions using all 4 models for a single matchup.
    Returns results with winner and winner probability.
    """

    # Create dataframes with correct feature order
    df_dt = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb1}])
    df_xgb2 = pd.DataFrame([{f: features.get(f, 0) for f in features_xgb2}])

    # Make predictions
    results = {
        'wrestler1': w1_name,
        'wrestler2': w2_name,
        'home_wrestler': home_name,
        'away_wrestler': away_name,
        'weight_class': weight_class
    }

    # LOGREG
    logreg_input = [[features['win_rate_diff']]]
    logreg_proba = logreg_model.predict_proba(logreg_input)[0]  # [away_prob, home_prob]
    logreg_pred = logreg_model.predict(logreg_input)[0]  # 1 = home wins, 0 = away wins

    results['logreg_pred'] = int(logreg_pred)
    results['logreg_home_prob'] = round(logreg_proba[1], 4)
    results['logreg_away_prob'] = round(logreg_proba[0], 4)
    results['logreg_winner'] = home_name if logreg_pred == 1 else away_name
    results['logreg_winner_prob'] = round(logreg_proba[1] if logreg_pred == 1 else logreg_proba[0], 4)

    # DT
    dt_input = df_dt[features_dt_xgb1]
    dt_proba = dt_model.predict_proba(dt_input)[0]
    dt_pred = dt_model.predict(dt_input)[0]

    results['dt_pred'] = int(dt_pred)
    results['dt_home_prob'] = round(dt_proba[1], 4)
    results['dt_away_prob'] = round(dt_proba[0], 4)
    results['dt_winner'] = home_name if dt_pred == 1 else away_name
    results['dt_winner_prob'] = round(dt_proba[1] if dt_pred == 1 else dt_proba[0], 4)

    # XGB1
    xgb1_input = df_dt[features_dt_xgb1]
    xgb1_proba = xgb_model1.predict_proba(xgb1_input)[0]
    xgb1_pred = xgb_model1.predict(xgb1_input)[0]

    results['xgb1_pred'] = int(xgb1_pred)
    results['xgb1_home_prob'] = round(xgb1_proba[1], 4)
    results['xgb1_away_prob'] = round(xgb1_proba[0], 4)
    results['xgb1_winner'] = home_name if xgb1_pred == 1 else away_name
    results['xgb1_winner_prob'] = round(xgb1_proba[1] if xgb1_pred == 1 else xgb1_proba[0], 4)

    # XGB2
    xgb2_input = df_xgb2[features_xgb2]
    xgb2_proba = xgb_model2.predict_proba(xgb2_input)[0]
    xgb2_pred = xgb_model2.predict(xgb2_input)[0]

    results['xgb2_pred'] = int(xgb2_pred)
    results['xgb2_home_prob'] = round(xgb2_proba[1], 4)
    results['xgb2_away_prob'] = round(xgb2_proba[0], 4)
    results['xgb2_winner'] = home_name if xgb2_pred == 1 else away_name
    results['xgb2_winner_prob'] = round(xgb2_proba[1] if xgb2_pred == 1 else xgb2_proba[0], 4)

    return results


# ============================================
# FUNCTION TO PROCESS A WEIGHT CLASS
# ============================================

def process_ncaa_weight_class(weight_class, wrestler_list, inference_db, matches_df,
                              logreg_model, dt_model, xgb_model1, xgb_model2,
                              features_dt_xgb1, features_xgb2):
    """
    Process all matchups for an NCAA weight class.
    Returns predictions DataFrame and champion info.
    """

    print("\n" + "="*100)
    print(f"🏋️  PROCESSING NCAA {weight_class}lbs WEIGHT CLASS")
    print("="*100)

    # Verify count
    print(f"\n📋 Wrestlers in {weight_class}lbs: {len(wrestler_list)}")

    # Generate all pairwise matchups
    total_matchups = len(list(combinations(wrestler_list, 2)))
    print(f"\n🔄 Generating predictions for {total_matchups} matchups...")

    weight_predictions = []
    matchup_count = 0

    for w1, w2 in combinations(wrestler_list, 2):
        matchup_count += 1
        if matchup_count % 50 == 0 or matchup_count == total_matchups:
            print(f"   Progress: {matchup_count}/{total_matchups} matchups")

        # Get historical data
        hist_features = get_wrestler_match_history(w1, w2, matches_df)

        # Prepare features
        features, home_name, away_name, home, away = prepare_matchup_features(
            w1, w2, hist_features, inference_db, weight_class=weight_class
        )

        # Make predictions
        results = predict_single_matchup(w1, w2, features, home_name, away_name, weight_class)
        weight_predictions.append(results)

    # Create dataframe
    weight_df = pd.DataFrame(weight_predictions)

    # Add model agreement column
    weight_df['all_agree'] = (
        (weight_df['logreg_winner'] == weight_df['dt_winner']) &
        (weight_df['dt_winner'] == weight_df['xgb1_winner']) &
        (weight_df['xgb1_winner'] == weight_df['xgb2_winner'])
    )

    # Calculate average confidence
    weight_df['avg_confidence'] = (
        weight_df['logreg_winner_prob'] +
        weight_df['dt_winner_prob'] +
        weight_df['xgb1_winner_prob'] +
        weight_df['xgb2_winner_prob']
    ) / 4

    # Determine predicted champions
    champions = {}
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        winner_col = f'{model}_winner'
        champ_counts = weight_df[winner_col].value_counts()
        champion = champ_counts.index[0]
        champ_wins = champ_counts.values[0]
        champions[model] = champion

    # Display summary
    print(f"\n📊 {weight_class}lbs - QUICK SUMMARY")
    print("-" * 60)
    print(f"Total Matchups: {len(weight_df)}")
    print(f"All models agree: {weight_df['all_agree'].sum()}/{len(weight_df)} ({weight_df['all_agree'].mean()*100:.1f}%)")

    # Top 3 most confident predictions
    top_confident = weight_df.nlargest(3, 'avg_confidence')
    print("\n🔝 Top 3 most confident predictions:")
    for _, row in top_confident.iterrows():
        print(f"   {row['home_wrestler']} vs {row['away_wrestler']}: {row['avg_confidence']*100:.1f}%")

    # Predicted champions
    print("\n🏆 Predicted champions:")
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        champ = champions[model]
        champ_wins = weight_df[weight_df[f'{model}_winner'] == champ].shape[0]
        print(f"   {model.upper():6}: {champ} ({champ_wins}/{len(wrestler_list)-1} wins, {champ_wins/(len(wrestler_list)-1)*100:.1f}%)")

    # Check for unanimous champion
    unique_champs = set(champions.values())
    if len(unique_champs) == 1:
        print(f"\n✅ UNANIMOUS CHAMPION: {list(unique_champs)[0]}")
    else:
        print(f"\n⚠️ No unanimous champion")
        for model, champ in champions.items():
            print(f"   {model.upper():6}: {champ}")

    return weight_df, champions

In [70]:
print("\n" + "="*100)
print("🎯 STARTING WITH 125lbs - 528 MATCHUPS")
print("="*100)

# Initialize storage
all_ncaa_predictions = {}
all_ncaa_champions = []

# Process 125lbs using inference_db (with Class column)
pred_125, champs_125 = process_ncaa_weight_class(
    125, wrestlers_125, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[125] = pred_125
# Add weight to champions dict
champs_125['weight'] = 125
all_ncaa_champions.append(champs_125)

# Save 125lbs results
#pred_125.to_csv('ncaa_125lbs_predictions.csv', index=False)
#print(f"\n💾 Saved 125lbs predictions to ncaa_125lbs_predictions.csv")

# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_125.head(10))

print(f"\n✅ 125lbs complete! {len(pred_125)} predictions generated")



🎯 STARTING WITH 125lbs - 528 MATCHUPS

🏋️  PROCESSING NCAA 125lbs WEIGHT CLASS

📋 Wrestlers in 125lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 125lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 338/528 (64.0%)

🔝 Top 3 most confident predictions:
   Luke Lilledahl vs Kael Lauridsen: 89.4%
   Sheldon Seymour vs Kael Lauridsen: 89.1%
   Luke Lilledahl vs Tyler Chappell: 88.5%

🏆 Predicted champions:
   LOGREG: Luke Lilledahl (32/32 wins, 100.0%)
   DT    : Luke Lilledahl (32/32 wins, 100.0%)
   XGB1  : Luke Lilledahl (32/32 wins, 100.0%)
   XGB2  : Luke Lilledahl (32/32 wins, 100.0%

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Jace Schafer,Mack Mauger,Mack Mauger,Jace Schafer,125,0,0.4392,0.5608,Jace Schafer,0.5608,...,0.7055,Jace Schafer,0.7055,1,0.7433,0.2567,Mack Mauger,0.7433,False,0.635150
1,Jace Schafer,Luke Lilledahl,Luke Lilledahl,Jace Schafer,125,1,0.7125,0.2875,Luke Lilledahl,0.7125,...,0.2530,Luke Lilledahl,0.7470,1,0.9150,0.0850,Luke Lilledahl,0.9150,True,0.780125
2,Jace Schafer,Jett Strickenberger,Jett Strickenberger,Jace Schafer,125,1,0.5091,0.4909,Jett Strickenberger,0.5091,...,0.5707,Jace Schafer,0.5707,1,0.7904,0.2096,Jett Strickenberger,0.7904,False,0.654325
3,Jace Schafer,Ezekiel Witt,Ezekiel Witt,Jace Schafer,125,0,0.4884,0.5116,Jace Schafer,0.5116,...,0.5153,Jace Schafer,0.5153,1,0.7795,0.2205,Ezekiel Witt,0.7795,False,0.584350
4,Jace Schafer,Maximo Renteria,Maximo Renteria,Jace Schafer,125,1,0.6447,0.3553,Maximo Renteria,0.6447,...,0.4241,Maximo Renteria,0.5759,1,0.8225,0.1775,Maximo Renteria,0.8225,True,0.697275
5,Jace Schafer,Ayden Smith,Ayden Smith,Jace Schafer,125,0,0.4851,0.5149,Jace Schafer,0.5149,...,0.5789,Jace Schafer,0.5789,1,0.7920,0.2080,Ayden Smith,0.7920,False,0.604200
6,Jace Schafer,Kael Lauridsen,Kael Lauridsen,Jace Schafer,125,0,0.3505,0.6495,Jace Schafer,0.6495,...,0.8835,Jace Schafer,0.8835,0,0.4363,0.5637,Jace Schafer,0.5637,True,0.738225
7,Jace Schafer,Dean Peterson,Dean Peterson,Jace Schafer,125,1,0.5707,0.4293,Dean Peterson,0.5707,...,0.4883,Dean Peterson,0.5117,1,0.8582,0.1418,Dean Peterson,0.8582,True,0.637925
8,Jace Schafer,Troy Spratley,Troy Spratley,Jace Schafer,125,1,0.6158,0.3842,Troy Spratley,0.6158,...,0.4170,Troy Spratley,0.5830,1,0.9059,0.0941,Troy Spratley,0.9059,True,0.712675
9,Jace Schafer,Andrew Binni,Andrew Binni,Jace Schafer,125,1,0.5091,0.4909,Andrew Binni,0.5091,...,0.6986,Jace Schafer,0.6986,1,0.7021,0.2979,Andrew Binni,0.7021,False,0.664225



✅ 125lbs complete! 528 predictions generated


In [71]:
# 133
# Process 133lbs using inference_db (with Class column)
pred_133, champs_133 = process_ncaa_weight_class(
    133, wrestlers_133, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[133] = pred_133
# Add weight to champions dict
champs_133['weight'] = 133
all_ncaa_champions.append(champs_133)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_133.head(10))

print(f"\n✅ 133lbs complete! {len(pred_133)} predictions generated")


🏋️  PROCESSING NCAA 133lbs WEIGHT CLASS

📋 Wrestlers in 133lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 133lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 326/528 (61.7%)

🔝 Top 3 most confident predictions:
   Marcus Blaze vs Carter Schmidt: 90.5%
   Ben Davino vs Carter Schmidt: 89.7%
   Jax Forrest vs Carter Schmidt: 87.8%

🏆 Predicted champions:
   LOGREG: Lucas Byrd (32/32 wins, 100.0%)
   DT    : Jax Forrest (32/32 wins, 100.0%)
   XGB1  : TK Davis (31/32 wins, 96.9%)
   XGB2  : Lucas Byrd (32/32 wins, 100.0%)

⚠️ No unanimous champion
   LOGREG: Lucas Byrd
   DT    : Jax Fo

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Carter Schmidt,Andrew Austin,Andrew Austin,Carter Schmidt,133,1,0.6897,0.3103,Andrew Austin,0.6897,...,0.1929,Andrew Austin,0.8071,1,0.7069,0.2931,Andrew Austin,0.7069,True,0.737425
1,Carter Schmidt,Jax Forrest,Jax Forrest,Carter Schmidt,133,1,0.8522,0.1478,Jax Forrest,0.8522,...,0.1564,Jax Forrest,0.8436,1,0.9276,0.0724,Jax Forrest,0.9276,True,0.878300
2,Carter Schmidt,TK Davis,TK Davis,Carter Schmidt,133,1,0.8522,0.1478,TK Davis,0.8522,...,0.1258,TK Davis,0.8742,1,0.7963,0.2037,TK Davis,0.7963,True,0.853125
3,Carter Schmidt,Zan Fugitt,Zan Fugitt,Carter Schmidt,133,1,0.7757,0.2243,Zan Fugitt,0.7757,...,0.1137,Zan Fugitt,0.8863,1,0.9207,0.0793,Zan Fugitt,0.9207,True,0.832175
4,Carter Schmidt,Dominick Serrano,Dominick Serrano,Carter Schmidt,133,1,0.8227,0.1773,Dominick Serrano,0.8227,...,0.1297,Dominick Serrano,0.8703,1,0.5668,0.4332,Dominick Serrano,0.5668,True,0.751450
5,Carter Schmidt,Blake Boarman,Blake Boarman,Carter Schmidt,133,1,0.6600,0.3400,Blake Boarman,0.6600,...,0.2786,Blake Boarman,0.7214,1,0.5876,0.4124,Blake Boarman,0.5876,True,0.678750
6,Carter Schmidt,Will Betancourt,Will Betancourt,Carter Schmidt,133,1,0.7557,0.2443,Will Betancourt,0.7557,...,0.1848,Will Betancourt,0.8152,1,0.8226,0.1774,Will Betancourt,0.8226,True,0.784875
7,Carter Schmidt,Markel Baker,Markel Baker,Carter Schmidt,133,1,0.8329,0.1671,Markel Baker,0.8329,...,0.1154,Markel Baker,0.8846,1,0.8589,0.1411,Markel Baker,0.8589,True,0.830600
8,Carter Schmidt,Kyler Larkin,Kyler Larkin,Carter Schmidt,133,1,0.8522,0.1478,Kyler Larkin,0.8522,...,0.1405,Kyler Larkin,0.8595,1,0.8743,0.1257,Kyler Larkin,0.8743,True,0.833000
9,Carter Schmidt,Garrett Grice,Garrett Grice,Carter Schmidt,133,1,0.8522,0.1478,Garrett Grice,0.8522,...,0.3017,Garrett Grice,0.6983,1,0.8058,0.1942,Garrett Grice,0.8058,True,0.775575



✅ 133lbs complete! 528 predictions generated


In [72]:
# Process 141lbs using inference_db (with Class column)
pred_141, champs_141 = process_ncaa_weight_class(
    141, wrestlers_141, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[141] = pred_141
# Add weight to champions dict
champs_141['weight'] = 141
all_ncaa_champions.append(champs_141)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_141.head(10))

print(f"\n✅ 141lbs complete! {len(pred_141)} predictions generated")


🏋️  PROCESSING NCAA 141lbs WEIGHT CLASS

📋 Wrestlers in 141lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 141lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 378/528 (71.6%)

🔝 Top 3 most confident predictions:
   Jesse Mendez vs Matthew Martino: 88.4%
   Sergio Vega vs Matthew Martino: 88.0%
   Jesse Mendez vs Tom Crook: 87.4%

🏆 Predicted champions:
   LOGREG: Jesse Mendez (32/32 wins, 100.0%)
   DT    : Jesse Mendez (32/32 wins, 100.0%)
   XGB1  : Jesse Mendez (32/32 wins, 100.0%)
   XGB2  : Jesse Mendez (32/32 wins, 100.0%)

✅ UNANIMOUS CHAMPION: Jesse Mendez

📋 First 10 predicti

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Aldo Hernandez,Matthew Martino,Matthew Martino,Aldo Hernandez,141,0,0.4607,0.5393,Aldo Hernandez,0.5393,...,0.5602,Aldo Hernandez,0.5602,1,0.6806,0.3194,Matthew Martino,0.6806,False,0.577775
1,Aldo Hernandez,Jesse Mendez,Jesse Mendez,Aldo Hernandez,141,1,0.7653,0.2347,Jesse Mendez,0.7653,...,0.1348,Jesse Mendez,0.8652,1,0.9391,0.0609,Jesse Mendez,0.9391,True,0.864850
2,Aldo Hernandez,Caedyn Ricciardi,Caedyn Ricciardi,Aldo Hernandez,141,1,0.6784,0.3216,Caedyn Ricciardi,0.6784,...,0.2429,Caedyn Ricciardi,0.7571,1,0.9133,0.0867,Caedyn Ricciardi,0.9133,True,0.773700
3,Aldo Hernandez,Ryan Jack,Ryan Jack,Aldo Hernandez,141,1,0.6122,0.3878,Ryan Jack,0.6122,...,0.1978,Ryan Jack,0.8022,1,0.9303,0.0697,Ryan Jack,0.9303,True,0.772675
4,Aldo Hernandez,Joey Olivieri,Joey Olivieri,Aldo Hernandez,141,1,0.7653,0.2347,Joey Olivieri,0.7653,...,0.1728,Joey Olivieri,0.8272,1,0.9285,0.0715,Joey Olivieri,0.9285,True,0.816750
5,Aldo Hernandez,Nash Singleton,Nash Singleton,Aldo Hernandez,141,1,0.5903,0.4097,Nash Singleton,0.5903,...,0.3763,Nash Singleton,0.6237,1,0.6328,0.3672,Nash Singleton,0.6328,True,0.614475
6,Aldo Hernandez,Tom Crook,Tom Crook,Aldo Hernandez,141,0,0.4781,0.5219,Aldo Hernandez,0.5219,...,0.3702,Tom Crook,0.6298,1,0.8916,0.1084,Tom Crook,0.8916,False,0.663600
7,Aldo Hernandez,Vance Vombaur,Vance Vombaur,Aldo Hernandez,141,1,0.6457,0.3543,Vance Vombaur,0.6457,...,0.2122,Vance Vombaur,0.7878,1,0.9159,0.0841,Vance Vombaur,0.9159,True,0.773850
8,Aldo Hernandez,Luke Stanich,Luke Stanich,Aldo Hernandez,141,1,0.7653,0.2347,Luke Stanich,0.7653,...,0.1683,Luke Stanich,0.8317,1,0.8953,0.1047,Luke Stanich,0.8953,True,0.845525
9,Aldo Hernandez,Pierson Manville,Pierson Manville,Aldo Hernandez,141,1,0.6251,0.3749,Pierson Manville,0.6251,...,0.2214,Pierson Manville,0.7786,1,0.8614,0.1386,Pierson Manville,0.8614,True,0.752775



✅ 141lbs complete! 528 predictions generated


In [73]:
# Process 149lbs using inference_db (with Class column)
pred_149, champs_149 = process_ncaa_weight_class(
    149, wrestlers_149, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[149] = pred_149
# Add weight to champions dict
champs_149['weight'] = 149
all_ncaa_champions.append(champs_149)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_149.head(10))

print(f"\n✅ 149lbs complete! {len(pred_149)} predictions generated")


🏋️  PROCESSING NCAA 149lbs WEIGHT CLASS

📋 Wrestlers in 149lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 149lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 331/528 (62.7%)

🔝 Top 3 most confident predictions:
   Shayne Van Ness vs Clayton Jones: 87.6%
   Jaxon Joy vs Clayton Jones: 87.3%
   Cross Wasilewski vs Clayton Jones: 86.9%

🏆 Predicted champions:
   LOGREG: Shayne Van Ness (32/32 wins, 100.0%)
   DT    : Shayne Van Ness (32/32 wins, 100.0%)
   XGB1  : Shayne Van Ness (32/32 wins, 100.0%)
   XGB2  : Shayne Van Ness (32/32 wins, 100.0%)

✅ UNANIMOUS CHAMPION: Shayne Van Ness


,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Austin McBurney,Clayton Jones,Clayton Jones,Austin McBurney,149,0,0.3973,0.6027,Austin McBurney,0.6027,...,0.7334,Austin McBurney,0.7334,0,0.1676,0.8324,Austin McBurney,0.8324,True,0.728900
1,Austin McBurney,Shayne Van Ness,Shayne Van Ness,Austin McBurney,149,1,0.7200,0.2800,Shayne Van Ness,0.7200,...,0.1597,Shayne Van Ness,0.8403,1,0.8984,0.1016,Shayne Van Ness,0.8984,True,0.837125
2,Austin McBurney,Lucas Kapusta,Lucas Kapusta,Austin McBurney,149,1,0.6581,0.3419,Lucas Kapusta,0.6581,...,0.2891,Lucas Kapusta,0.7109,1,0.8171,0.1829,Lucas Kapusta,0.8171,True,0.733025
3,Austin McBurney,Jacob Frost,Jacob Frost,Austin McBurney,149,1,0.5987,0.4013,Jacob Frost,0.5987,...,0.3630,Jacob Frost,0.6370,1,0.8915,0.1085,Jacob Frost,0.8915,True,0.718300
4,Austin McBurney,David Evans,David Evans,Austin McBurney,149,1,0.6741,0.3259,David Evans,0.6741,...,0.2742,David Evans,0.7258,1,0.8242,0.1758,David Evans,0.8242,True,0.742525
5,Austin McBurney,Andrew Clark,Andrew Clark,Austin McBurney,149,1,0.5659,0.4341,Andrew Clark,0.5659,...,0.4590,Andrew Clark,0.5410,1,0.8739,0.1261,Andrew Clark,0.8739,False,0.627950
6,Austin McBurney,Michael Gioffre,Michael Gioffre,Austin McBurney,149,0,0.4640,0.5360,Austin McBurney,0.5360,...,0.6209,Austin McBurney,0.6209,1,0.7384,0.2616,Michael Gioffre,0.7384,False,0.626600
7,Austin McBurney,Casey Swiderski,Casey Swiderski,Austin McBurney,149,1,0.5379,0.4621,Casey Swiderski,0.5379,...,0.6156,Austin McBurney,0.6156,1,0.7412,0.2588,Casey Swiderski,0.7412,False,0.626450
8,Austin McBurney,Koy Buesgens,Koy Buesgens,Austin McBurney,149,1,0.6654,0.3346,Koy Buesgens,0.6654,...,0.2628,Koy Buesgens,0.7372,1,0.9185,0.0815,Koy Buesgens,0.9185,True,0.766775
9,Austin McBurney,Kade Brown,Kade Brown,Austin McBurney,149,1,0.5416,0.4584,Kade Brown,0.5416,...,0.4321,Kade Brown,0.5679,1,0.7969,0.2031,Kade Brown,0.7969,True,0.629375



✅ 149lbs complete! 528 predictions generated


In [74]:
# Process 157lbs using inference_db (with Class column)
pred_157, champs_157 = process_ncaa_weight_class(
    157, wrestlers_157, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[157] = pred_157
# Add weight to champions dict
champs_157['weight'] = 157
all_ncaa_champions.append(champs_157)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_157.head(10))

print(f"\n✅ 157lbs complete! {len(pred_157)} predictions generated")


🏋️  PROCESSING NCAA 157lbs WEIGHT CLASS

📋 Wrestlers in 157lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 157lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 383/528 (72.5%)

🔝 Top 3 most confident predictions:
   Meyer Shapiro vs Yannis Charles: 90.7%
   PJ Duke vs Yannis Charles: 90.3%
   Landon Robideau vs Yannis Charles: 90.3%

🏆 Predicted champions:
   LOGREG: Meyer Shapiro (32/32 wins, 100.0%)
   DT    : Meyer Shapiro (32/32 wins, 100.0%)
   XGB1  : Meyer Shapiro (32/32 wins, 100.0%)
   XGB2  : Brandon Cannon (32/32 wins, 100.0%)

⚠️ No unanimous champion
   LOGREG: Meyer Shapi

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Yannis Charles,Jeb Prechtel,Jeb Prechtel,Yannis Charles,157,1,0.6387,0.3613,Jeb Prechtel,0.6387,...,0.4936,Jeb Prechtel,0.5064,1,0.8258,0.1742,Jeb Prechtel,0.8258,False,0.625475
1,Yannis Charles,PJ Duke,PJ Duke,Yannis Charles,157,1,0.8467,0.1533,PJ Duke,0.8467,...,0.0983,PJ Duke,0.9017,1,0.9752,0.0248,PJ Duke,0.9752,True,0.903350
2,Yannis Charles,Luke Mechler,Luke Mechler,Yannis Charles,157,1,0.7749,0.2251,Luke Mechler,0.7749,...,0.1343,Luke Mechler,0.8657,1,0.9631,0.0369,Luke Mechler,0.9631,True,0.837425
3,Yannis Charles,Cael Swensen,Cael Swensen,Yannis Charles,157,1,0.8326,0.1674,Cael Swensen,0.8326,...,0.1110,Cael Swensen,0.8890,1,0.9703,0.0297,Cael Swensen,0.9703,True,0.895425
4,Yannis Charles,Daniel Cardenas,Daniel Cardenas,Yannis Charles,157,1,0.8195,0.1805,Daniel Cardenas,0.8195,...,0.0924,Daniel Cardenas,0.9076,1,0.9706,0.0294,Daniel Cardenas,0.9706,True,0.896875
5,Yannis Charles,Jaivon Jones,Jaivon Jones,Yannis Charles,157,1,0.7321,0.2679,Jaivon Jones,0.7321,...,0.2616,Jaivon Jones,0.7384,1,0.9184,0.0816,Jaivon Jones,0.9184,True,0.783725
6,Yannis Charles,Mason Shrader,Mason Shrader,Yannis Charles,157,1,0.8086,0.1914,Mason Shrader,0.8086,...,0.0993,Mason Shrader,0.9007,1,0.9612,0.0388,Mason Shrader,0.9612,True,0.890075
7,Yannis Charles,Brandon Cannon,Brandon Cannon,Yannis Charles,157,1,0.8672,0.1328,Brandon Cannon,0.8672,...,0.1361,Brandon Cannon,0.8639,1,0.9511,0.0489,Brandon Cannon,0.9511,True,0.893000
8,Yannis Charles,Landon Robideau,Landon Robideau,Yannis Charles,157,1,0.8495,0.1505,Landon Robideau,0.8495,...,0.0996,Landon Robideau,0.9004,1,0.9730,0.0270,Landon Robideau,0.9730,True,0.903175
9,Yannis Charles,Gavin Drexler,Gavin Drexler,Yannis Charles,157,1,0.7321,0.2679,Gavin Drexler,0.7321,...,0.2042,Gavin Drexler,0.7958,1,0.9497,0.0503,Gavin Drexler,0.9497,True,0.805900



✅ 157lbs complete! 528 predictions generated


In [75]:
# Process 165lbs using inference_db (with Class column)
pred_165, champs_165 = process_ncaa_weight_class(
    165, wrestlers_165, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[165] = pred_165
# Add weight to champions dict
champs_165['weight'] = 165
all_ncaa_champions.append(champs_165)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_165.head(10))

print(f"\n✅ 165lbs complete! {len(pred_165)} predictions generated")


🏋️  PROCESSING NCAA 165lbs WEIGHT CLASS

📋 Wrestlers in 165lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 165lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 291/528 (55.1%)

🔝 Top 3 most confident predictions:
   Mitchell Mesenbrink vs Ryan Vigil: 86.5%
   Mitchell Mesenbrink vs EJ Parco: 86.2%
   Joey Blaze vs Ryan Vigil: 86.2%

🏆 Predicted champions:
   LOGREG: Mitchell Mesenbrink (32/32 wins, 100.0%)
   DT    : Mitchell Mesenbrink (32/32 wins, 100.0%)
   XGB1  : Mitchell Mesenbrink (32/32 wins, 100.0%)
   XGB2  : Mitchell Mesenbrink (32/32 wins, 100.0%)

✅ UNANIMOUS CHAMPION: Mit

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Ryan Vigil,Cody Walsh,Cody Walsh,Ryan Vigil,165,1,0.6492,0.3508,Cody Walsh,0.6492,...,0.2564,Cody Walsh,0.7436,1,0.9117,0.0883,Cody Walsh,0.9117,True,0.762625
1,Ryan Vigil,Mitchell Mesenbrink,Mitchell Mesenbrink,Ryan Vigil,165,1,0.7722,0.2278,Mitchell Mesenbrink,0.7722,...,0.1394,Mitchell Mesenbrink,0.8606,1,0.9378,0.0622,Mitchell Mesenbrink,0.9378,True,0.865100
2,Ryan Vigil,Braeden Scoles,Braeden Scoles,Ryan Vigil,165,1,0.6322,0.3678,Braeden Scoles,0.6322,...,0.2500,Braeden Scoles,0.7500,1,0.9255,0.0745,Braeden Scoles,0.9255,True,0.763425
3,Ryan Vigil,Paddy Gallagher,Paddy Gallagher,Ryan Vigil,165,1,0.5535,0.4465,Paddy Gallagher,0.5535,...,0.3636,Paddy Gallagher,0.6364,1,0.8778,0.1222,Paddy Gallagher,0.8778,True,0.669700
4,Ryan Vigil,Bryce Hepner,Bryce Hepner,Ryan Vigil,165,1,0.6055,0.3945,Bryce Hepner,0.6055,...,0.4070,Bryce Hepner,0.5930,1,0.8829,0.1171,Bryce Hepner,0.8829,True,0.706850
5,Ryan Vigil,Sean Seefeldt,Sean Seefeldt,Ryan Vigil,165,1,0.6382,0.3618,Sean Seefeldt,0.6382,...,0.3482,Sean Seefeldt,0.6518,1,0.9181,0.0819,Sean Seefeldt,0.9181,True,0.738525
6,Ryan Vigil,Mac Church,Mac Church,Ryan Vigil,165,1,0.5712,0.4288,Mac Church,0.5712,...,0.5361,Ryan Vigil,0.5361,1,0.8673,0.1327,Mac Church,0.8673,False,0.646425
7,Ryan Vigil,Matt Bianchi,Matt Bianchi,Ryan Vigil,165,1,0.7414,0.2586,Matt Bianchi,0.7414,...,0.3106,Matt Bianchi,0.6894,1,0.9119,0.0881,Matt Bianchi,0.9119,True,0.772175
8,Ryan Vigil,Ladarion Lockett,Ladarion Lockett,Ryan Vigil,165,1,0.7414,0.2586,Ladarion Lockett,0.7414,...,0.3549,Ladarion Lockett,0.6451,1,0.9197,0.0803,Ladarion Lockett,0.9197,True,0.763050
9,Ryan Vigil,Cody Goebel,Cody Goebel,Ryan Vigil,165,1,0.5330,0.4670,Cody Goebel,0.5330,...,0.4283,Cody Goebel,0.5717,1,0.9034,0.0966,Cody Goebel,0.9034,False,0.634775



✅ 165lbs complete! 528 predictions generated


In [76]:
# Process 174lbs using inference_db (with Class column)
pred_174, champs_174 = process_ncaa_weight_class(
    174, wrestlers_174, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[174] = pred_174
# Add weight to champions dict
champs_174['weight'] = 174
all_ncaa_champions.append(champs_174)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_174.head(10))

print(f"\n✅ 174lbs complete! {len(pred_174)} predictions generated")


🏋️  PROCESSING NCAA 174lbs WEIGHT CLASS

📋 Wrestlers in 174lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 174lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 353/528 (66.9%)

🔝 Top 3 most confident predictions:
   Simon Ruiz vs Collin Carrigan: 90.4%
   Levi Haines vs Collin Carrigan: 89.3%
   Patrick Kennedy vs Collin Carrigan: 89.2%

🏆 Predicted champions:
   LOGREG: Levi Haines (32/32 wins, 100.0%)
   DT    : Levi Haines (32/32 wins, 100.0%)
   XGB1  : Levi Haines (32/32 wins, 100.0%)
   XGB2  : Levi Haines (32/32 wins, 100.0%)

✅ UNANIMOUS CHAMPION: Levi Haines

📋 First 10 predic

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Grant O'Dell,Lucas Condon,Lucas Condon,Grant O'Dell,174,0,0.3544,0.6456,Grant O'Dell,0.6456,...,0.6202,Grant O'Dell,0.6202,1,0.6635,0.3365,Lucas Condon,0.6635,False,0.657325
1,Grant O'Dell,Levi Haines,Levi Haines,Grant O'Dell,174,1,0.6639,0.3361,Levi Haines,0.6639,...,0.1381,Levi Haines,0.8619,1,0.9444,0.0556,Levi Haines,0.9444,True,0.840000
2,Grant O'Dell,Jared Simma,Jared Simma,Grant O'Dell,174,0,0.4661,0.5339,Grant O'Dell,0.5339,...,0.4258,Jared Simma,0.5742,1,0.8569,0.1431,Jared Simma,0.8569,False,0.644025
3,Grant O'Dell,Nick Fine,Nick Fine,Grant O'Dell,174,1,0.5444,0.4556,Nick Fine,0.5444,...,0.3805,Nick Fine,0.6195,1,0.7921,0.2079,Nick Fine,0.7921,True,0.641775
4,Grant O'Dell,Beau Mantanona,Beau Mantanona,Grant O'Dell,174,1,0.5248,0.4752,Beau Mantanona,0.5248,...,0.4020,Beau Mantanona,0.5980,1,0.8981,0.1019,Beau Mantanona,0.8981,True,0.658000
5,Grant O'Dell,Garrett Thompson,Garrett Thompson,Grant O'Dell,174,1,0.5492,0.4508,Garrett Thompson,0.5492,...,0.3378,Garrett Thompson,0.6622,0,0.4354,0.5646,Grant O'Dell,0.5646,False,0.596775
6,Grant O'Dell,Sergio Desiante,Sergio Desiante,Grant O'Dell,174,0,0.3045,0.6955,Grant O'Dell,0.6955,...,0.7549,Grant O'Dell,0.7549,1,0.6692,0.3308,Sergio Desiante,0.6692,False,0.704900
7,Grant O'Dell,Alex Facundo,Alex Facundo,Grant O'Dell,174,1,0.5004,0.4996,Alex Facundo,0.5004,...,0.4148,Alex Facundo,0.5852,1,0.9173,0.0827,Alex Facundo,0.9173,True,0.653500
8,Grant O'Dell,Patrick Kennedy,Patrick Kennedy,Grant O'Dell,174,1,0.6008,0.3992,Patrick Kennedy,0.6008,...,0.2760,Patrick Kennedy,0.7240,1,0.8998,0.1002,Patrick Kennedy,0.8998,True,0.742650
9,Grant O'Dell,Holden Garcia,Holden Garcia,Grant O'Dell,174,0,0.4721,0.5279,Grant O'Dell,0.5279,...,0.4508,Holden Garcia,0.5492,1,0.7926,0.2074,Holden Garcia,0.7926,False,0.620200



✅ 174lbs complete! 528 predictions generated


In [77]:
# Process 184lbs using inference_db (with Class column)
pred_184, champs_184 = process_ncaa_weight_class(
    184, wrestlers_184, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[184] = pred_184
# Add weight to champions dict
champs_184['weight'] = 184
all_ncaa_champions.append(champs_184)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_184.head(10))

print(f"\n✅ 184lbs complete! {len(pred_184)} predictions generated")


🏋️  PROCESSING NCAA 184lbs WEIGHT CLASS

📋 Wrestlers in 184lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 184lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 304/528 (57.6%)

🔝 Top 3 most confident predictions:
   Rocco Welsh vs Mahonri Rushton: 87.1%
   Aeoden Sinclair vs Mahonri Rushton: 85.7%
   Max McEnelly vs Mahonri Rushton: 85.6%

🏆 Predicted champions:
   LOGREG: Rocco Welsh (32/32 wins, 100.0%)
   DT    : Joe Curtis (32/32 wins, 100.0%)
   XGB1  : Rocco Welsh (31/32 wins, 96.9%)
   XGB2  : Rocco Welsh (32/32 wins, 100.0%)

⚠️ No unanimous champion
   LOGREG: Rocco Welsh
   D

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Sam Goin,Caleb Uhlenhopp,Sam Goin,Caleb Uhlenhopp,184,1,0.5444,0.4556,Sam Goin,0.5444,...,0.4294,Sam Goin,0.5706,1,0.6511,0.3489,Sam Goin,0.6511,True,0.594300
1,Sam Goin,Rocco Welsh,Rocco Welsh,Sam Goin,184,1,0.7003,0.2997,Rocco Welsh,0.7003,...,0.1460,Rocco Welsh,0.8540,1,0.9609,0.0391,Rocco Welsh,0.9609,True,0.815300
2,Sam Goin,Ian Bush,Ian Bush,Sam Goin,184,0,0.4944,0.5056,Sam Goin,0.5056,...,0.2859,Ian Bush,0.7141,1,0.9000,0.1000,Ian Bush,0.9000,False,0.682700
3,Sam Goin,Rylan Rogers,Sam Goin,Rylan Rogers,184,0,0.4629,0.5371,Rylan Rogers,0.5371,...,0.6292,Rylan Rogers,0.6292,0,0.1941,0.8059,Rylan Rogers,0.8059,True,0.679825
4,Sam Goin,Chris Moore,Chris Moore,Sam Goin,184,1,0.5080,0.4920,Chris Moore,0.5080,...,0.2958,Chris Moore,0.7042,1,0.9102,0.0898,Chris Moore,0.9102,True,0.683375
5,Sam Goin,Joe Curtis,Sam Goin,Joe Curtis,184,0,0.3040,0.6960,Joe Curtis,0.6960,...,0.6377,Joe Curtis,0.6377,0,0.2634,0.7366,Joe Curtis,0.7366,True,0.731625
6,Sam Goin,Malachi Duvall,Sam Goin,Malachi Duvall,184,0,0.3475,0.6525,Malachi Duvall,0.6525,...,0.8565,Malachi Duvall,0.8565,0,0.1568,0.8432,Malachi Duvall,0.8432,True,0.802100
7,Sam Goin,Silas Allred,Silas Allred,Sam Goin,184,1,0.5200,0.4800,Silas Allred,0.5200,...,0.2946,Silas Allred,0.7054,1,0.9425,0.0575,Silas Allred,0.9425,True,0.694750
8,Sam Goin,Brock Mantanona,Brock Mantanona,Sam Goin,184,1,0.5664,0.4336,Brock Mantanona,0.5664,...,0.2872,Brock Mantanona,0.7128,1,0.8789,0.1211,Brock Mantanona,0.8789,True,0.692300
9,Sam Goin,Abe Wojcikiewicz,Sam Goin,Abe Wojcikiewicz,184,1,0.5444,0.4556,Sam Goin,0.5444,...,0.4097,Sam Goin,0.5903,0,0.2701,0.7299,Abe Wojcikiewicz,0.7299,False,0.618925



✅ 184lbs complete! 528 predictions generated


In [78]:
# Process 197lbs using inference_db (with Class column)
pred_197, champs_197 = process_ncaa_weight_class(
    197, wrestlers_197, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[197] = pred_197
# Add weight to champions dict
champs_197['weight'] = 197
all_ncaa_champions.append(champs_197)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_197.head(10))

print(f"\n✅ 197lbs complete! {len(pred_197)} predictions generated")


🏋️  PROCESSING NCAA 197lbs WEIGHT CLASS

📋 Wrestlers in 197lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 197lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 289/528 (54.7%)

🔝 Top 3 most confident predictions:
   Josh Barr vs Kael Bennie: 87.4%
   Zayne Lehman vs Kael Bennie: 86.5%
   Justin Rademacher vs Kael Bennie: 85.8%

🏆 Predicted champions:
   LOGREG: Josh Barr (32/32 wins, 100.0%)
   DT    : Josh Barr (32/32 wins, 100.0%)
   XGB1  : Josh Barr (32/32 wins, 100.0%)
   XGB2  : Josh Barr (32/32 wins, 100.0%)

✅ UNANIMOUS CHAMPION: Josh Barr

📋 First 10 predictions:


,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Karson Tompkins,Blake Schaffer,Karson Tompkins,Blake Schaffer,197,0,0.4846,0.5154,Blake Schaffer,0.5154,...,0.5435,Blake Schaffer,0.5435,1,0.5647,0.4353,Karson Tompkins,0.5647,False,0.592675
1,Karson Tompkins,Josh Barr,Josh Barr,Karson Tompkins,197,1,0.6600,0.3400,Josh Barr,0.6600,...,0.1684,Josh Barr,0.8316,1,0.9360,0.0640,Josh Barr,0.9360,True,0.829350
2,Karson Tompkins,Dillon Bechtold,Dillon Bechtold,Karson Tompkins,197,1,0.5297,0.4703,Dillon Bechtold,0.5297,...,0.2182,Dillon Bechtold,0.7818,1,0.9022,0.0978,Dillon Bechtold,0.9022,True,0.706200
3,Karson Tompkins,Branson John,Branson John,Karson Tompkins,197,1,0.5102,0.4898,Branson John,0.5102,...,0.1908,Branson John,0.8092,1,0.7400,0.2600,Branson John,0.7400,True,0.667625
4,Karson Tompkins,Angelo Posada,Angelo Posada,Karson Tompkins,197,1,0.5744,0.4256,Angelo Posada,0.5744,...,0.3026,Angelo Posada,0.6974,1,0.9086,0.0914,Angelo Posada,0.9086,True,0.697875
5,Karson Tompkins,Brock Zurawski,Brock Zurawski,Karson Tompkins,197,1,0.5744,0.4256,Brock Zurawski,0.5744,...,0.2231,Brock Zurawski,0.7769,1,0.8925,0.1075,Brock Zurawski,0.8925,True,0.713725
6,Karson Tompkins,Evan Bates,Evan Bates,Karson Tompkins,197,0,0.4846,0.5154,Karson Tompkins,0.5154,...,0.2106,Evan Bates,0.7894,1,0.9609,0.0391,Evan Bates,0.9609,False,0.719200
7,Karson Tompkins,Deanthony Parker,Deanthony Parker,Karson Tompkins,197,0,0.4982,0.5018,Karson Tompkins,0.5018,...,0.2113,Deanthony Parker,0.7887,1,0.9209,0.0791,Deanthony Parker,0.9209,False,0.705625
8,Karson Tompkins,Joey Novak,Joey Novak,Karson Tompkins,197,1,0.5871,0.4129,Joey Novak,0.5871,...,0.3444,Joey Novak,0.6556,1,0.9176,0.0824,Joey Novak,0.9176,True,0.692850
9,Karson Tompkins,Kael Wisler,Kael Wisler,Karson Tompkins,197,0,0.4982,0.5018,Karson Tompkins,0.5018,...,0.2500,Kael Wisler,0.7500,1,0.6377,0.3623,Kael Wisler,0.6377,False,0.625150



✅ 197lbs complete! 528 predictions generated


In [79]:
# Process 197lbs using inference_db (with Class column)
pred_285, champs_285 = process_ncaa_weight_class(
    285, wrestlers_285, inference_db_with_class, matches,
    logreg_model, dt_model, xgb_model1, xgb_model2,
    features_dt_xgb1, features_xgb2
)

# Store results
all_ncaa_predictions[285] = pred_285
# Add weight to champions dict
champs_285['weight'] = 285
all_ncaa_champions.append(champs_285)


# Display first few rows
print("\n📋 First 10 predictions:")
display(pred_285.head(10))

print(f"\n✅ 285lbs complete! {len(pred_285)} predictions generated")


🏋️  PROCESSING NCAA 285lbs WEIGHT CLASS

📋 Wrestlers in 285lbs: 33

🔄 Generating predictions for 528 matchups...
   Progress: 50/528 matchups
   Progress: 100/528 matchups
   Progress: 150/528 matchups
   Progress: 200/528 matchups
   Progress: 250/528 matchups
   Progress: 300/528 matchups
   Progress: 350/528 matchups
   Progress: 400/528 matchups
   Progress: 450/528 matchups
   Progress: 500/528 matchups
   Progress: 528/528 matchups

📊 285lbs - QUICK SUMMARY
------------------------------------------------------------
Total Matchups: 528
All models agree: 315/528 (59.7%)

🔝 Top 3 most confident predictions:
   Taye Ghadiali vs Jarrett Stoner: 86.8%
   Yonger Bastida vs Jarrett Stoner: 86.7%
   Isaac Trumble vs Jarrett Stoner: 86.0%

🏆 Predicted champions:
   LOGREG: Yonger Bastida (32/32 wins, 100.0%)
   DT    : Yonger Bastida (32/32 wins, 100.0%)
   XGB1  : Yonger Bastida (32/32 wins, 100.0%)
   XGB2  : Yonger Bastida (32/32 wins, 100.0%)

✅ UNANIMOUS CHAMPION: Yonger Bastida

📋

,wrestler1,wrestler2,home_wrestler,away_wrestler,weight_class,logreg_pred,logreg_home_prob,logreg_away_prob,logreg_winner,logreg_winner_prob,...,xgb1_away_prob,xgb1_winner,xgb1_winner_prob,xgb2_pred,xgb2_home_prob,xgb2_away_prob,xgb2_winner,xgb2_winner_prob,all_agree,avg_confidence
0,Mason Rebuck,Emmanuel Ulrich,Emmanuel Ulrich,Mason Rebuck,285,0,0.3864,0.6136,Mason Rebuck,0.6136,...,0.7903,Mason Rebuck,0.7903,0,0.3548,0.6452,Mason Rebuck,0.6452,True,0.699050
1,Mason Rebuck,Yonger Bastida,Yonger Bastida,Mason Rebuck,285,1,0.6352,0.3648,Yonger Bastida,0.6352,...,0.3285,Yonger Bastida,0.6715,1,0.8473,0.1527,Yonger Bastida,0.8473,True,0.725000
2,Mason Rebuck,Vincent Mueller,Vincent Mueller,Mason Rebuck,285,1,0.5476,0.4524,Vincent Mueller,0.5476,...,0.5693,Mason Rebuck,0.5693,1,0.7829,0.2171,Vincent Mueller,0.7829,False,0.661725
3,Mason Rebuck,Jim Mullen,Jim Mullen,Mason Rebuck,285,0,0.4295,0.5705,Mason Rebuck,0.5705,...,0.6632,Mason Rebuck,0.6632,1,0.7971,0.2029,Jim Mullen,0.7971,False,0.694475
4,Mason Rebuck,Cole Mirasola,Cole Mirasola,Mason Rebuck,285,0,0.4933,0.5067,Mason Rebuck,0.5067,...,0.3980,Cole Mirasola,0.6020,1,0.8160,0.1840,Cole Mirasola,0.8160,False,0.633950
5,Mason Rebuck,Connor Barket,Connor Barket,Mason Rebuck,285,1,0.5297,0.4703,Connor Barket,0.5297,...,0.4151,Connor Barket,0.5849,1,0.5526,0.4474,Connor Barket,0.5526,True,0.569575
6,Mason Rebuck,Alex Semenenko,Alex Semenenko,Mason Rebuck,285,1,0.5546,0.4454,Alex Semenenko,0.5546,...,0.5168,Mason Rebuck,0.5168,1,0.6646,0.3354,Alex Semenenko,0.6646,False,0.620775
7,Mason Rebuck,Ben Kueter,Ben Kueter,Mason Rebuck,285,0,0.3695,0.6305,Mason Rebuck,0.6305,...,0.8041,Mason Rebuck,0.8041,0,0.4578,0.5422,Mason Rebuck,0.5422,True,0.669200
8,Mason Rebuck,Nick Feldman,Nick Feldman,Mason Rebuck,285,1,0.5107,0.4893,Nick Feldman,0.5107,...,0.3926,Nick Feldman,0.6074,1,0.8966,0.1034,Nick Feldman,0.8966,True,0.656450
9,Mason Rebuck,Jarrett Stoner,Jarrett Stoner,Mason Rebuck,285,0,0.3343,0.6657,Mason Rebuck,0.6657,...,0.8699,Mason Rebuck,0.8699,1,0.5818,0.4182,Jarrett Stoner,0.5818,False,0.743400



✅ 285lbs complete! 528 predictions generated


In [80]:
# ============================================
# ASSEMBLE MASTER PREDICTIONS DATAFRAME
# ============================================

print("\n" + "="*100)
print("📊 ASSEMBLING MASTER PREDICTIONS DATAFRAME")
print("="*100)

# Combine all weight class predictions
master_pred_df = pd.concat([
    pred_125, pred_133, pred_141, pred_149, pred_157,
    pred_165, pred_174, pred_184, pred_197, pred_285
], ignore_index=True)

print(f"✅ Master predictions dataframe created!")
print(f"   Total rows: {len(master_pred_df)}")
print(f"   Columns: {master_pred_df.columns.tolist()}")
print(f"   Weight classes: {master_pred_df['weight_class'].unique()}")

# Save master predictions
master_pred_df.to_csv('ncaa_master_predictions.csv', index=False)
print("\n💾 Saved: ncaa_master_predictions.csv")

# ============================================
# CREATE CHAMPIONS DATAFRAME WITH PROBABILITIES
# ============================================

print("\n" + "="*100)
print("🏆 CREATING CHAMPIONS DATAFRAME WITH PROBABILITIES")
print("="*100)

champions_data = []

for weight in [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]:
    weight_df = all_ncaa_predictions[weight]

    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        winner_col = f'{model}_winner'
        prob_col = f'{model}_winner_prob'

        # Get champion for this model
        champ_counts = weight_df[winner_col].value_counts()
        champion = champ_counts.index[0]
        champ_wins = champ_counts.values[0]
        total_possible = 32  # 33 wrestlers - 1

        # Calculate average win probability for the champion
        champ_matchups = weight_df[weight_df[winner_col] == champion]
        avg_prob = champ_matchups[prob_col].mean()
        min_prob = champ_matchups[prob_col].min()
        max_prob = champ_matchups[prob_col].max()

        # Find who they're predicted to lose to (if any)
        predicted_losses = []
        if champ_wins < total_possible:
            # They didn't win all matches
            for _, row in weight_df.iterrows():
                if row[winner_col] != champion:
                    if row['home_wrestler'] == champion:
                        opponent = row['away_wrestler']
                        prob = row[prob_col]  # This is probability of champion winning
                        predicted_losses.append({
                            'opponent': opponent,
                            'probability': 1 - prob  # Probability of losing
                        })
                    elif row['away_wrestler'] == champion:
                        opponent = row['home_wrestler']
                        prob = row[prob_col]
                        predicted_losses.append({
                            'opponent': opponent,
                            'probability': 1 - prob
                        })

        champions_data.append({
            'weight_class': weight,
            'model': model.upper(),
            'champion': champion,
            'wins': champ_wins,
            'total_possible': total_possible,
            'win_percentage': f"{champ_wins/total_possible*100:.1f}%",
            'avg_win_prob': f"{avg_prob*100:.1f}%",
            'min_win_prob': f"{min_prob*100:.1f}%",
            'max_win_prob': f"{max_prob*100:.1f}%",
            'predicted_losses': len(predicted_losses),
            'loss_details': str(predicted_losses) if predicted_losses else 'None'
        })

champions_df = pd.DataFrame(champions_data)
print("\n🏆 Champions by Model:")
display(champions_df[['weight_class', 'model', 'champion', 'wins', 'win_percentage', 'avg_win_prob', 'predicted_losses']])

# Save champions dataframe
# champions_df.to_csv('ncaa_champions.csv', index=False)
# print("\n💾 Saved: ncaa_champions.csv")

# ============================================
# FIXED: CREATE TABLEAU-FRIENDLY HEAD-TO-HEAD DATAFRAME
# ============================================

print("\n" + "="*100)
print("📊 CREATING TABLEAU-FRIENDLY HEAD-TO-HEAD DATAFRAME (FIXED)")
print("="*100)

tableau_rows = []

for weight in [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]:
    weight_df = all_ncaa_predictions[weight]

    print(f"\nProcessing {weight}lbs: {len(weight_df)} matchups")

    for _, row in weight_df.iterrows():
        w1 = row['wrestler1']
        w2 = row['wrestler2']
        home = row['home_wrestler']
        away = row['away_wrestler']

        # For each matchup, create rows for both wrestlers as wrestler_a
        for wrestler_a in [w1, w2]:
            opponent = w2 if wrestler_a == w1 else w1

            # For each model's prediction
            for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
                winner = row[f'{model}_winner']
                prob = row[f'{model}_winner_prob']

                # Determine if wrestler_a wins or loses
                if winner == wrestler_a:
                    winner_prob = prob
                    loser_prob = 1 - prob
                    predicted_winner = wrestler_a
                    predicted_loser = opponent
                else:
                    winner_prob = 1 - prob
                    loser_prob = prob
                    predicted_winner = opponent
                    predicted_loser = wrestler_a

                # Determine home/away status for wrestler_a
                is_home = (wrestler_a == home)

                tableau_rows.append({
                    'weight_class': weight,
                    'wrestler_a': wrestler_a,
                    'wrestler_b': opponent,
                    'model': model.upper(),
                    'predicted_winner': predicted_winner,
                    'winner_probability': round(winner_prob * 100, 1),
                    'predicted_loser': predicted_loser,
                    'loser_probability': round(loser_prob * 100, 1),
                    'wrestler_a_is_home': is_home,
                    'home_wrestler': home,
                    'away_wrestler': away,
                    'all_models_agree': row['all_agree'],
                    'avg_confidence': round(row['avg_confidence'] * 100, 1)
                })

tableau_df = pd.DataFrame(tableau_rows)
print(f"\n✅ Tableau dataframe created with {len(tableau_df)} rows")
print(f"   Each matchup now has 2×4 = 8 rows (2 wrestlers × 4 models)")

# ============================================
# VERIFY THE FIX
# ============================================

print("\n" + "="*100)
print("🔍 VERIFYING THE FIX")
print("="*100)

test_wrestlers = ['Anthony Echemendia', 'Sergio Vega', 'Drake Ayala']

for wrestler in test_wrestlers:
    unique_opponents = tableau_df[tableau_df['wrestler_a'] == wrestler]['wrestler_b'].nunique()
    total_possible = len(tableau_df[tableau_df['wrestler_a'] == wrestler]) / 4  # Divide by 4 models
    print(f"\n📊 {wrestler}:")
    print(f"   Unique opponents: {unique_opponents}")
    print(f"   Total matchups: {total_possible:.0f}")

    # Check if they have 32 opponents (33 wrestlers - themselves)
    if unique_opponents == 32:
        print(f"   ✅ CORRECT: Has all 32 possible opponents")
    else:
        print(f"   ❌ ERROR: Missing {32 - unique_opponents} opponents")

# ============================================
# DISPLAY SAMPLE
# ============================================

print("\n" + "="*100)
print("📋 SAMPLE ROWS (showing both perspectives of same matchup)")
print("="*100)

# Pick a specific matchup to demonstrate
sample_match = tableau_df[
    (tableau_df['wrestler_a'] == 'Luke Lilledahl') &
    (tableau_df['wrestler_b'] == 'Jace Schafer')
].head(4)  # 4 models for Luke's perspective

sample_match2 = tableau_df[
    (tableau_df['wrestler_a'] == 'Jace Schafer') &
    (tableau_df['wrestler_b'] == 'Luke Lilledahl')
].head(4)  # 4 models for Jace's perspective

display(pd.concat([sample_match, sample_match2]))

# ============================================
# SUMMARY STATISTICS
# ============================================

print("\n" + "="*100)
print("📊 SUMMARY STATISTICS")
print("="*100)

summary = []
for weight in [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]:
    weight_wrestlers = tableau_df[tableau_df['weight_class'] == weight]['wrestler_a'].unique()
    weight_matchups = len(tableau_df[tableau_df['weight_class'] == weight]) / 4  # Divide by 4 models

    summary.append({
        'Weight': f"{weight}lbs",
        'Wrestlers': len(weight_wrestlers),
        'Total Matchup Rows': len(tableau_df[tableau_df['weight_class'] == weight]),
        'Unique Matchups': weight_matchups,
        'Expected Matchups': len(weight_wrestlers) * (len(weight_wrestlers) - 1)
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

# ============================================
# SAVE FIXED DATAFRAME
# ============================================

tableau_df.to_csv('ncaa_head_to_head_tableau_fixed.csv', index=False)
print("\n💾 Saved: ncaa_head_to_head_tableau_fixed.csv")

print("\n✅ FIX COMPLETE - Each wrestler now has all 32 opponents!")


📊 ASSEMBLING MASTER PREDICTIONS DATAFRAME
✅ Master predictions dataframe created!
   Total rows: 5280
   Columns: ['wrestler1', 'wrestler2', 'home_wrestler', 'away_wrestler', 'weight_class', 'logreg_pred', 'logreg_home_prob', 'logreg_away_prob', 'logreg_winner', 'logreg_winner_prob', 'dt_pred', 'dt_home_prob', 'dt_away_prob', 'dt_winner', 'dt_winner_prob', 'xgb1_pred', 'xgb1_home_prob', 'xgb1_away_prob', 'xgb1_winner', 'xgb1_winner_prob', 'xgb2_pred', 'xgb2_home_prob', 'xgb2_away_prob', 'xgb2_winner', 'xgb2_winner_prob', 'all_agree', 'avg_confidence']
   Weight classes: [125 133 141 149 157 165 174 184 197 285]

💾 Saved: ncaa_master_predictions.csv

🏆 CREATING CHAMPIONS DATAFRAME WITH PROBABILITIES

🏆 Champions by Model:


,weight_class,model,champion,wins,win_percentage,avg_win_prob,predicted_losses
0,125,LOGREG,Luke Lilledahl,32,100.0%,65.7%,0
1,125,DT,Luke Lilledahl,32,100.0%,74.3%,0
2,125,XGB1,Luke Lilledahl,32,100.0%,76.3%,0
3,125,XGB2,Luke Lilledahl,32,100.0%,84.5%,0
4,133,LOGREG,Lucas Byrd,32,100.0%,63.4%,0
5,133,DT,Jax Forrest,32,100.0%,76.3%,0
6,133,XGB1,TK Davis,31,96.9%,79.3%,1
7,133,XGB2,Lucas Byrd,32,100.0%,80.2%,0
8,141,LOGREG,Jesse Mendez,32,100.0%,65.9%,0
9,141,DT,Jesse Mendez,32,100.0%,79.7%,0



📊 CREATING TABLEAU-FRIENDLY HEAD-TO-HEAD DATAFRAME (FIXED)

Processing 125lbs: 528 matchups

Processing 133lbs: 528 matchups

Processing 141lbs: 528 matchups

Processing 149lbs: 528 matchups

Processing 157lbs: 528 matchups

Processing 165lbs: 528 matchups

Processing 174lbs: 528 matchups

Processing 184lbs: 528 matchups

Processing 197lbs: 528 matchups

Processing 285lbs: 528 matchups

✅ Tableau dataframe created with 42240 rows
   Each matchup now has 2×4 = 8 rows (2 wrestlers × 4 models)

🔍 VERIFYING THE FIX

📊 Anthony Echemendia:
   Unique opponents: 32
   Total matchups: 32
   ✅ CORRECT: Has all 32 possible opponents

📊 Sergio Vega:
   Unique opponents: 32
   Total matchups: 32
   ✅ CORRECT: Has all 32 possible opponents

📊 Drake Ayala:
   Unique opponents: 32
   Total matchups: 32
   ✅ CORRECT: Has all 32 possible opponents

📋 SAMPLE ROWS (showing both perspectives of same matchup)


,weight_class,wrestler_a,wrestler_b,model,predicted_winner,winner_probability,predicted_loser,loser_probability,wrestler_a_is_home,home_wrestler,away_wrestler,all_models_agree,avg_confidence
12,125,Luke Lilledahl,Jace Schafer,LOGREG,Luke Lilledahl,71.2,Jace Schafer,28.7,True,Luke Lilledahl,Jace Schafer,True,78.0
13,125,Luke Lilledahl,Jace Schafer,DT,Luke Lilledahl,74.6,Jace Schafer,25.4,True,Luke Lilledahl,Jace Schafer,True,78.0
14,125,Luke Lilledahl,Jace Schafer,XGB1,Luke Lilledahl,74.7,Jace Schafer,25.3,True,Luke Lilledahl,Jace Schafer,True,78.0
15,125,Luke Lilledahl,Jace Schafer,XGB2,Luke Lilledahl,91.5,Jace Schafer,8.5,True,Luke Lilledahl,Jace Schafer,True,78.0
8,125,Jace Schafer,Luke Lilledahl,LOGREG,Luke Lilledahl,28.7,Jace Schafer,71.2,False,Luke Lilledahl,Jace Schafer,True,78.0
9,125,Jace Schafer,Luke Lilledahl,DT,Luke Lilledahl,25.4,Jace Schafer,74.6,False,Luke Lilledahl,Jace Schafer,True,78.0
10,125,Jace Schafer,Luke Lilledahl,XGB1,Luke Lilledahl,25.3,Jace Schafer,74.7,False,Luke Lilledahl,Jace Schafer,True,78.0
11,125,Jace Schafer,Luke Lilledahl,XGB2,Luke Lilledahl,8.5,Jace Schafer,91.5,False,Luke Lilledahl,Jace Schafer,True,78.0



📊 SUMMARY STATISTICS


,Weight,Wrestlers,Total Matchup Rows,Unique Matchups,Expected Matchups
0,125lbs,33,4224,1056.0,1056
1,133lbs,33,4224,1056.0,1056
2,141lbs,33,4224,1056.0,1056
3,149lbs,33,4224,1056.0,1056
4,157lbs,33,4224,1056.0,1056
5,165lbs,33,4224,1056.0,1056
6,174lbs,33,4224,1056.0,1056
7,184lbs,33,4224,1056.0,1056
8,197lbs,33,4224,1056.0,1056
9,285lbs,33,4224,1056.0,1056



💾 Saved: ncaa_head_to_head_tableau_fixed.csv

✅ FIX COMPLETE - Each wrestler now has all 32 opponents!


In [81]:
get_head_to_head('Anthony Echemendia', 'Aldo Hernandez', master_pred_df, 141)

{'weight_class': np.int64(141),
 'matchup': 'Anthony Echemendia vs Aldo Hernandez',
 'results': {'logreg': {'winner': 'Anthony Echemendia',
   'probability': np.float64(0.6868),
   'home_advantage': True},
  'dt': {'winner': 'Anthony Echemendia',
   'probability': np.float64(0.8898),
   'home_advantage': True},
  'xgb1': {'winner': 'Anthony Echemendia',
   'probability': np.float32(0.8072),
   'home_advantage': True},
  'xgb2': {'winner': 'Anthony Echemendia',
   'probability': np.float32(0.9059),
   'home_advantage': True},
  'voting': {'winner': 'Anthony Echemendia',
   'probability': np.float64(0.8275000005086263),
   'home_advantage': True}},
 'all_agree': np.True_,
 'avg_confidence': np.float64(0.822425004029274)}

In [82]:
# ============================================
# FUNCTION TO ANALYZE MODEL DISAGREEMENTS
# ============================================

def analyze_champion_disagreements(weight_class, master_df, champions_df):
    """
    Analyze why models disagree on champion predictions for a weight class.
    """
    print("\n" + "="*80)
    print(f"🔍 ANALYZING {weight_class}lbs CHAMPION DISAGREEMENTS")
    print("="*80)

    # Get champions for this weight
    weight_champs = champions_df[champions_df['weight_class'] == weight_class]

    print(f"\n🏆 Champions by model:")
    for _, row in weight_champs.iterrows():
        print(f"   {row['model']}: {row['champion']} ({row['wins']}/{row['total_possible']} wins, {row['win_percentage']})")

    # Find unique champions
    unique_champs = weight_champs['champion'].unique()
    print(f"\n📊 Unique champions: {', '.join(unique_champs)}")

    # Analyze each unique champion
    for champ in unique_champs:
        print(f"\n🔎 Analyzing: {champ}")

        # Get all matchups involving this wrestler
        weight_df = master_df[master_df['weight_class'] == weight_class]
        champ_matchups = weight_df[
            (weight_df['home_wrestler'] == champ) |
            (weight_df['away_wrestler'] == champ)
        ].copy()

        print(f"   Total matchups: {len(champ_matchups)}")

        # Check each model's predictions for this wrestler
        for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
            winner_col = f'{model}_winner'
            prob_col = f'{model}_winner_prob'

            # Count wins for this wrestler according to this model
            model_wins = champ_matchups[champ_matchups[winner_col] == champ].shape[0]
            model_pct = (model_wins / len(champ_matchups)) * 100

            # Find who they're predicted to lose to (if any)
            losses = champ_matchups[champ_matchups[winner_col] != champ]

            print(f"\n   {model.upper()}: {model_wins}/{len(champ_matchups)} wins ({model_pct:.1f}%)")

            if len(losses) > 0:
                print(f"      Predicted losses:")
                for _, loss in losses.iterrows():
                    if loss['home_wrestler'] == champ:
                        opponent = loss['away_wrestler']
                        prob = loss[prob_col]  # Probability of champ winning
                    else:
                        opponent = loss['home_wrestler']
                        prob = loss[prob_col]

                    print(f"         vs {opponent}: {prob*100:.1f}% chance to win ({(1-prob)*100:.1f}% chance to lose)")
            else:
                print(f"      No predicted losses (perfect record)")


# ============================================
# FUNCTION TO GET HEAD-TO-HEAD PREDICTIONS
# ============================================

def get_head_to_head(wrestler1, wrestler2, master_df, weight_class=None):
    """
    Get all model predictions for a specific head-to-head matchup.
    """
    print("\n" + "="*80)
    print(f"🔍 HEAD-TO-HEAD: {wrestler1} vs {wrestler2}")
    print("="*80)

    # Find the matchup
    mask = (
        ((master_df['home_wrestler'] == wrestler1) & (master_df['away_wrestler'] == wrestler2)) |
        ((master_df['home_wrestler'] == wrestler2) & (master_df['away_wrestler'] == wrestler1))
    )

    if weight_class:
        mask = mask & (master_df['weight_class'] == weight_class)

    matchup = master_df[mask]

    if len(matchup) == 0:
        print(f"❌ No matchup found between {wrestler1} and {wrestler2}")
        return None

    row = matchup.iloc[0]
    weight = row['weight_class']

    print(f"\n📊 Weight Class: {weight}lbs")
    print(f"\n{'Model':<15} {'Predicted Winner':<25} {'Confidence':<15} {'Winner Prob':<15}")
    print("-" * 70)

    results = {}
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        winner = row[f'{model}_winner']
        prob = row[f'{model}_winner_prob']

        # Get home/away info
        if row['home_wrestler'] == winner:
            home_away = "(Home)"
        else:
            home_away = "(Away)"

        print(f"{model.upper():<15} {winner:25} {prob*100:>6.1f}% {home_away:15}")

        results[model] = {
            'winner': winner,
            'probability': prob,
            'home_advantage': row['home_wrestler'] == winner
        }

    print(f"\n📈 Models agree: {'✅' if row['all_agree'] else '❌'}")
    print(f"📊 Average confidence: {row['avg_confidence']*100:.1f}%")

    return results


# ============================================
# ANALYZE INTERESTING CASES
# ============================================

# Analyze 133lbs disagreements
print("\n" + "="*100)
print("🔬 ANALYZING 133lbs DISAGREEMENTS")
print("="*100)

analyze_champion_disagreements(133, master_pred_df, champions_df)

# Analyze 184lbs disagreements
print("\n" + "="*100)
print("🔬 ANALYZING 184lbs DISAGREEMENTS")
print("="*100)

analyze_champion_disagreements(184, master_pred_df, champions_df)

# Test head-to-head function on TK Davis vs someone
print("\n" + "="*100)
print("🔬 TESTING HEAD-TO-HEAD FUNCTION")
print("="*100)

# For 133lbs, find who TK Davis is predicted to lose to
get_head_to_head('TK Davis', 'Ben Davino', master_pred_df, 133)
get_head_to_head('TK Davis', 'Jax Forrest', master_pred_df, 133)
get_head_to_head('TK Davis', 'Lucas Byrd', master_pred_df, 133)

# For 184lbs, analyze Rocco Welsh vs Joe Curtis
get_head_to_head('Rocco Welsh', 'Joe Curtis', master_pred_df, 184)


🔬 ANALYZING 133lbs DISAGREEMENTS

🔍 ANALYZING 133lbs CHAMPION DISAGREEMENTS

🏆 Champions by model:
   LOGREG: Lucas Byrd (32/32 wins, 100.0%)
   DT: Jax Forrest (32/32 wins, 100.0%)
   XGB1: TK Davis (31/32 wins, 96.9%)
   XGB2: Lucas Byrd (32/32 wins, 100.0%)

📊 Unique champions: Lucas Byrd, Jax Forrest, TK Davis

🔎 Analyzing: Lucas Byrd
   Total matchups: 32

   LOGREG: 32/32 wins (100.0%)
      No predicted losses (perfect record)

   DT: 27/32 wins (84.4%)
      Predicted losses:
         vs Jax Forrest: 53.1% chance to win (46.9% chance to lose)
         vs TK Davis: 53.1% chance to win (46.9% chance to lose)
         vs Aaron Seidel: 53.1% chance to win (46.9% chance to lose)
         vs Marcus Blaze: 53.1% chance to win (46.9% chance to lose)
         vs Ben Davino: 53.1% chance to win (46.9% chance to lose)

   XGB1: 30/32 wins (93.8%)
      Predicted losses:
         vs TK Davis: 54.9% chance to win (45.1% chance to lose)
         vs Ben Davino: 52.3% chance to win (47.7% cha

{'logreg': {'winner': 'Rocco Welsh',
  'probability': np.float64(0.5025),
  'home_advantage': True},
 'dt': {'winner': 'Joe Curtis',
  'probability': np.float64(0.531),
  'home_advantage': False},
 'xgb1': {'winner': 'Joe Curtis',
  'probability': np.float32(0.5142),
  'home_advantage': False},
 'xgb2': {'winner': 'Rocco Welsh',
  'probability': np.float32(0.7652),
  'home_advantage': True}}

In [83]:
# ============================================
# CREATE CHAMPIONS DATAFRAME - PIVOT STYLE
# ============================================

print("\n" + "="*100)
print("🏆 CREATING CHAMPIONS DATAFRAME - PIVOT STYLE")
print("="*100)

champions_pivot_data = []

for weight in [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]:
    weight_df = all_ncaa_predictions[weight]

    # Dictionary to store champion info for this weight class
    weight_row = {
        'weight_class': weight,
        'total_wrestlers': 33,
        'total_matchups': len(weight_df)
    }

    # For each model, get champion and their stats
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        model_upper = model.upper()
        winner_col = f'{model}_winner'
        prob_col = f'{model}_winner_prob'

        # Get champion for this model
        champ_counts = weight_df[winner_col].value_counts()
        champion = champ_counts.index[0]
        champ_wins = champ_counts.values[0]
        total_possible = 32

        # Calculate average win probability for the champion
        champ_matchups = weight_df[weight_df[winner_col] == champion]
        avg_prob = champ_matchups[prob_col].mean()
        min_prob = champ_matchups[prob_col].min()
        max_prob = champ_matchups[prob_col].max()

        # Store in columns
        weight_row[f'{model_upper}_champion'] = champion
        weight_row[f'{model_upper}_wins'] = f"{champ_wins}/{total_possible}"
        weight_row[f'{model_upper}_win_pct'] = f"{champ_wins/total_possible*100:.1f}%"
        weight_row[f'{model_upper}_avg_prob'] = f"{avg_prob*100:.1f}%"
        weight_row[f'{model_upper}_prob_range'] = f"{min_prob*100:.1f}%-{max_prob*100:.1f}%"

        # Check if unanimous for this model (champion wins all matches)
        weight_row[f'{model_upper}_unanimous'] = '✅' if champ_wins == total_possible else '❌'

    # Check if all models agree on the same champion
    champions = [
        weight_row['LOGREG_champion'],
        weight_row['DT_champion'],
        weight_row['XGB1_champion'],
        weight_row['XGB2_champion']
    ]
    weight_row['all_models_agree'] = '✅' if all(c == champions[0] for c in champions) else '❌'

    # Count how many models agree
    from collections import Counter
    champ_counter = Counter(champions)
    most_common_champ, count = champ_counter.most_common(1)[0]
    weight_row['consensus_champion'] = most_common_champ
    weight_row['models_agreeing'] = f"{count}/4"

    champions_pivot_data.append(weight_row)

# Create dataframe
champions_pivot_df = pd.DataFrame(champions_pivot_data)

# Reorder columns for better readability
column_order = [
    'weight_class', 'total_wrestlers', 'total_matchups',
    'LOGREG_champion', 'LOGREG_wins', 'LOGREG_win_pct', 'LOGREG_avg_prob', 'LOGREG_unanimous',
    'DT_champion', 'DT_wins', 'DT_win_pct', 'DT_avg_prob', 'DT_unanimous',
    'XGB1_champion', 'XGB1_wins', 'XGB1_win_pct', 'XGB1_avg_prob', 'XGB1_unanimous',
    'XGB2_champion', 'XGB2_wins', 'XGB2_win_pct', 'XGB2_avg_prob', 'XGB2_unanimous',
    'consensus_champion', 'models_agreeing', 'all_models_agree'
]

# Only include columns that exist
available_cols = [col for col in column_order if col in champions_pivot_df.columns]
champions_pivot_df = champions_pivot_df[available_cols]

print("\n🏆 CHAMPIONS BY WEIGHT CLASS - PIVOT VIEW")
print("="*100)
display(champions_pivot_df)

# Save champions dataframe
# champions_pivot_df.to_csv('ncaa_champions_pivot.csv', index=False)
# print("\n💾 Saved: ncaa_champions_pivot.csv")

# ============================================
# CREATE A SIMPLER SUMMARY VIEW
# ============================================

print("\n" + "="*100)
print("📊 SIMPLIFIED CHAMPIONS SUMMARY")
print("="*100)

simplified = champions_pivot_df[[
    'weight_class',
    'LOGREG_champion', 'DT_champion', 'XGB1_champion', 'XGB2_champion',
    'consensus_champion', 'models_agreeing', 'all_models_agree'
]].copy()

simplified.columns = [
    'Weight',
    'LOGREG', 'DT', 'XGB1', 'XGB2',
    'Consensus', 'Agreement', 'Unanimous'
]

display(simplified)

# ============================================
# CREATE A PROBABILITY MATRIX FOR TOP CONTENDERS
# ============================================

print("\n" + "="*100)
print("📊 CHAMPION WIN PROBABILITY MATRIX")
print("="*100)

# For each weight class, show the champion's win probability range
prob_matrix = champions_pivot_df[[
    'weight_class',
    'LOGREG_champion', 'LOGREG_avg_prob',
    'DT_champion', 'DT_avg_prob',
    'XGB1_champion', 'XGB1_avg_prob',
    'XGB2_champion', 'XGB2_avg_prob'
]].copy()

prob_matrix.columns = [
    'Weight',
    'LOGREG Champ', 'LOGREG Avg Prob',
    'DT Champ', 'DT Avg Prob',
    'XGB1 Champ', 'XGB1 Avg Prob',
    'XGB2 Champ', 'XGB2 Avg Prob'
]

display(prob_matrix)


🏆 CREATING CHAMPIONS DATAFRAME - PIVOT STYLE

🏆 CHAMPIONS BY WEIGHT CLASS - PIVOT VIEW


,weight_class,total_wrestlers,total_matchups,LOGREG_champion,LOGREG_wins,LOGREG_win_pct,LOGREG_avg_prob,LOGREG_unanimous,DT_champion,DT_wins,...,XGB1_avg_prob,XGB1_unanimous,XGB2_champion,XGB2_wins,XGB2_win_pct,XGB2_avg_prob,XGB2_unanimous,consensus_champion,models_agreeing,all_models_agree
0,125,33,528,Luke Lilledahl,32/32,100.0%,65.7%,✅,Luke Lilledahl,32/32,...,76.3%,✅,Luke Lilledahl,32/32,100.0%,84.5%,✅,Luke Lilledahl,4/4,✅
1,133,33,528,Lucas Byrd,32/32,100.0%,63.4%,✅,Jax Forrest,32/32,...,79.3%,❌,Lucas Byrd,32/32,100.0%,80.2%,✅,Lucas Byrd,2/4,❌
2,141,33,528,Jesse Mendez,32/32,100.0%,65.9%,✅,Jesse Mendez,32/32,...,80.9%,✅,Jesse Mendez,32/32,100.0%,86.0%,✅,Jesse Mendez,4/4,✅
3,149,33,528,Shayne Van Ness,32/32,100.0%,64.5%,✅,Shayne Van Ness,32/32,...,76.8%,✅,Shayne Van Ness,32/32,100.0%,84.2%,✅,Shayne Van Ness,4/4,✅
4,157,33,528,Meyer Shapiro,32/32,100.0%,64.9%,✅,Meyer Shapiro,32/32,...,77.2%,✅,Brandon Cannon,32/32,100.0%,80.6%,✅,Meyer Shapiro,3/4,❌
5,165,33,528,Mitchell Mesenbrink,32/32,100.0%,65.0%,✅,Mitchell Mesenbrink,32/32,...,78.1%,✅,Mitchell Mesenbrink,32/32,100.0%,88.5%,✅,Mitchell Mesenbrink,4/4,✅
6,174,33,528,Levi Haines,32/32,100.0%,65.7%,✅,Levi Haines,32/32,...,80.7%,✅,Levi Haines,32/32,100.0%,87.1%,✅,Levi Haines,4/4,✅
7,184,33,528,Rocco Welsh,32/32,100.0%,64.4%,✅,Joe Curtis,32/32,...,74.3%,❌,Rocco Welsh,32/32,100.0%,85.4%,✅,Rocco Welsh,3/4,❌
8,197,33,528,Josh Barr,32/32,100.0%,63.1%,✅,Josh Barr,32/32,...,76.1%,✅,Josh Barr,32/32,100.0%,83.9%,✅,Josh Barr,4/4,✅
9,285,33,528,Yonger Bastida,32/32,100.0%,63.6%,✅,Yonger Bastida,32/32,...,71.6%,✅,Yonger Bastida,32/32,100.0%,81.6%,✅,Yonger Bastida,4/4,✅



📊 SIMPLIFIED CHAMPIONS SUMMARY


,Weight,LOGREG,DT,XGB1,XGB2,Consensus,Agreement,Unanimous
0,125,Luke Lilledahl,Luke Lilledahl,Luke Lilledahl,Luke Lilledahl,Luke Lilledahl,4/4,✅
1,133,Lucas Byrd,Jax Forrest,TK Davis,Lucas Byrd,Lucas Byrd,2/4,❌
2,141,Jesse Mendez,Jesse Mendez,Jesse Mendez,Jesse Mendez,Jesse Mendez,4/4,✅
3,149,Shayne Van Ness,Shayne Van Ness,Shayne Van Ness,Shayne Van Ness,Shayne Van Ness,4/4,✅
4,157,Meyer Shapiro,Meyer Shapiro,Meyer Shapiro,Brandon Cannon,Meyer Shapiro,3/4,❌
5,165,Mitchell Mesenbrink,Mitchell Mesenbrink,Mitchell Mesenbrink,Mitchell Mesenbrink,Mitchell Mesenbrink,4/4,✅
6,174,Levi Haines,Levi Haines,Levi Haines,Levi Haines,Levi Haines,4/4,✅
7,184,Rocco Welsh,Joe Curtis,Rocco Welsh,Rocco Welsh,Rocco Welsh,3/4,❌
8,197,Josh Barr,Josh Barr,Josh Barr,Josh Barr,Josh Barr,4/4,✅
9,285,Yonger Bastida,Yonger Bastida,Yonger Bastida,Yonger Bastida,Yonger Bastida,4/4,✅



📊 CHAMPION WIN PROBABILITY MATRIX


,Weight,LOGREG Champ,LOGREG Avg Prob,DT Champ,DT Avg Prob,XGB1 Champ,XGB1 Avg Prob,XGB2 Champ,XGB2 Avg Prob
0,125,Luke Lilledahl,65.7%,Luke Lilledahl,74.3%,Luke Lilledahl,76.3%,Luke Lilledahl,84.5%
1,133,Lucas Byrd,63.4%,Jax Forrest,76.3%,TK Davis,79.3%,Lucas Byrd,80.2%
2,141,Jesse Mendez,65.9%,Jesse Mendez,79.7%,Jesse Mendez,80.9%,Jesse Mendez,86.0%
3,149,Shayne Van Ness,64.5%,Shayne Van Ness,76.3%,Shayne Van Ness,76.8%,Shayne Van Ness,84.2%
4,157,Meyer Shapiro,64.9%,Meyer Shapiro,74.5%,Meyer Shapiro,77.2%,Brandon Cannon,80.6%
5,165,Mitchell Mesenbrink,65.0%,Mitchell Mesenbrink,78.4%,Mitchell Mesenbrink,78.1%,Mitchell Mesenbrink,88.5%
6,174,Levi Haines,65.7%,Levi Haines,76.1%,Levi Haines,80.7%,Levi Haines,87.1%
7,184,Rocco Welsh,64.4%,Joe Curtis,76.4%,Rocco Welsh,74.3%,Rocco Welsh,85.4%
8,197,Josh Barr,63.1%,Josh Barr,77.2%,Josh Barr,76.1%,Josh Barr,83.9%
9,285,Yonger Bastida,63.6%,Yonger Bastida,73.9%,Yonger Bastida,71.6%,Yonger Bastida,81.6%


# Predict Braket Style

In [84]:
# ============================================
# NCAA BRACKET PREDICTION SYSTEM
# ============================================


# ============================================
# HELPER FUNCTION TO GET MATCHUP PREDICTIONS
# ============================================

def get_matchup_prediction(w1, w2, weight_class, master_df, model='xgb2'):
    """
    Get prediction for a specific matchup from master predictions.
    Returns (winner, probability)
    """
    matchup = master_df[
        ((master_df['home_wrestler'] == w1) & (master_df['away_wrestler'] == w2)) |
        ((master_df['home_wrestler'] == w2) & (master_df['away_wrestler'] == w1))
    ]

    if len(matchup) == 0:
        print(f"⚠️ Warning: No prediction found for {w1} vs {w2}")
        return None, 0.5

    row = matchup.iloc[0]
    return row[f'{model}_winner'], row[f'{model}_winner_prob']


# ============================================
# VOTING CLASSIFIER
# ============================================

def voting_classifier(w1, w2, weight_class, master_df):
    """
    Combine LOGREG, DT, and XGB2 predictions (excludes XGB1).
    Returns (winner, avg_confidence)
    """
    models = ['logreg', 'dt', 'xgb2']
    votes = []
    confidences = []

    for model in models:
        winner, prob = get_matchup_prediction(w1, w2, weight_class, master_df, model)
        if winner:
            votes.append(winner)
            confidences.append(prob)

    if not votes:
        return None, 0.5

    # Count votes
    vote_counts = Counter(votes)
    winner = vote_counts.most_common(1)[0][0]

    # Average confidence for winner
    avg_conf = np.mean([confidences[i] for i, v in enumerate(votes) if v == winner])

    return winner, avg_conf


# ============================================
# BRACKET DEFINITIONS
# ============================================

class Bracket:
    """Represents an NCAA wrestling bracket for a weight class"""

    def __init__(self, weight_class, wrestlers_list):
        self.weight_class = weight_class
        self.wrestlers = wrestlers_list
        self.rounds = {
            'pigtail': [],
            'round_of_32': [],
            'round_of_16': [],
            'quarterfinals': [],
            'semifinals': [],
            'finals': []
        }
        self.results = {}

    def add_pigtail(self, wrestler1, wrestler2):
        """Add pigtail match (33 vs 32)"""
        self.rounds['pigtail'] = [{'w1': wrestler1, 'w2': wrestler2, 'winner': None}]

    def add_round_of_32(self, matchups):
        """
        Add round of 32 matchups.
        matchups: list of (w1, w2) tuples, with 'PIGTAIL_WINNER' placeholder for #1 seed's opponent
        """
        self.rounds['round_of_32'] = [{'w1': w1, 'w2': w2, 'winner': None} for w1, w2 in matchups]

    def simulate(self, master_df, model='xgb2', verbose=True):
        """Simulate the entire bracket for a given model"""

        if verbose:
            print(f"\n{'='*60}")
            print(f"🏆 {self.weight_class}lbs - {model.upper()} MODEL")
            print(f"{'='*60}")

        # ===== PIGTAIL ROUND =====
        if verbose:
            print("\n🔹 PIGTAIL ROUND")
            print("-" * 40)

        pigtail = self.rounds['pigtail'][0]
        w1, w2 = pigtail['w1'], pigtail['w2']

        if model == 'voting':
            winner, prob = voting_classifier(w1, w2, self.weight_class, master_df)
        else:
            winner, prob = get_matchup_prediction(w1, w2, self.weight_class, master_df, model)

        pigtail['winner'] = winner
        if verbose:
            print(f"{w1} vs {w2}: {winner} wins ({prob*100:.1f}%)")

        pigtail_winner = winner

        # ===== ROUND OF 32 =====
        if verbose:
            print("\n🔹 ROUND OF 32")
            print("-" * 40)

        round_of_32_winners = []

        for matchup in self.rounds['round_of_32']:
            w1, w2 = matchup['w1'], matchup['w2']

            # Handle #1 seed vs pigtail winner
            if w2 == 'PIGTAIL_WINNER':
                if verbose:
                    print(f"{w1} vs {pigtail_winner}: ", end="")

                if model == 'voting':
                    winner, prob = voting_classifier(w1, pigtail_winner, self.weight_class, master_df)
                else:
                    winner, prob = get_matchup_prediction(w1, pigtail_winner, self.weight_class, master_df, model)

                if verbose:
                    print(f"{winner} wins ({prob*100:.1f}%)")

                matchup['winner'] = winner
                round_of_32_winners.append(winner)
                continue

            # Regular matchup
            if model == 'voting':
                winner, prob = voting_classifier(w1, w2, self.weight_class, master_df)
            else:
                winner, prob = get_matchup_prediction(w1, w2, self.weight_class, master_df, model)

            if verbose:
                print(f"{w1} vs {w2}: {winner} wins ({prob*100:.1f}%)")

            matchup['winner'] = winner
            round_of_32_winners.append(winner)

        # ===== ROUND OF 16 =====
        if verbose:
            print("\n🔹 ROUND OF 16")
            print("-" * 40)

        round_of_16_winners = []
        for i in range(0, len(round_of_32_winners), 2):
            w1, w2 = round_of_32_winners[i], round_of_32_winners[i+1]

            if model == 'voting':
                winner, prob = voting_classifier(w1, w2, self.weight_class, master_df)
            else:
                winner, prob = get_matchup_prediction(w1, w2, self.weight_class, master_df, model)

            if verbose:
                print(f"{w1} vs {w2}: {winner} wins ({prob*100:.1f}%)")

            round_of_16_winners.append(winner)

        # ===== QUARTERFINALS =====
        if verbose:
            print("\n🔹 QUARTERFINALS")
            print("-" * 40)

        quarterfinal_winners = []
        for i in range(0, len(round_of_16_winners), 2):
            w1, w2 = round_of_16_winners[i], round_of_16_winners[i+1]

            if model == 'voting':
                winner, prob = voting_classifier(w1, w2, self.weight_class, master_df)
            else:
                winner, prob = get_matchup_prediction(w1, w2, self.weight_class, master_df, model)

            if verbose:
                print(f"{w1} vs {w2}: {winner} wins ({prob*100:.1f}%)")

            quarterfinal_winners.append(winner)

        # ===== SEMIFINALS =====
        if verbose:
            print("\n🔹 SEMIFINALS")
            print("-" * 40)

        semifinal_winners = []
        for i in range(0, len(quarterfinal_winners), 2):
            w1, w2 = quarterfinal_winners[i], quarterfinal_winners[i+1]

            if model == 'voting':
                winner, prob = voting_classifier(w1, w2, self.weight_class, master_df)
            else:
                winner, prob = get_matchup_prediction(w1, w2, self.weight_class, master_df, model)

            if verbose:
                print(f"{w1} vs {w2}: {winner} wins ({prob*100:.1f}%)")

            semifinal_winners.append(winner)

        # ===== FINALS =====
        if verbose:
            print("\n🔹 FINALS")
            print("-" * 40)

        w1, w2 = semifinal_winners[0], semifinal_winners[1]

        if model == 'voting':
            champion, prob = voting_classifier(w1, w2, self.weight_class, master_df)
        else:
            champion, prob = get_matchup_prediction(w1, w2, self.weight_class, master_df, model)

        if verbose:
            print(f"{w1} vs {w2}: {champion} wins ({prob*100:.1f}%)")
            print("\n" + "="*40)
            print(f"🏆 CHAMPION: {champion}")
            print("="*40)

        # Store results
        self.results[model] = {
            'pigtail_winner': pigtail_winner,
            'round_of_32_winners': round_of_32_winners,
            'round_of_16_winners': round_of_16_winners,
            'quarterfinal_winners': quarterfinal_winners,
            'semifinal_winners': semifinal_winners,
            'champion': champion,
            'champion_prob': prob
        }

        return self.results[model]

In [85]:
# ============================================
# CREATE 125lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 125lbs BRACKET")
print("="*100)

bracket_125 = Bracket(125, wrestlers_125)

# Pigtail match (33 vs 32)
bracket_125.add_pigtail('Jace Schafer', 'Mack Mauger')

# Round of 32 matchups (in order)
round_of_32_125 = [
    ('Luke Lilledahl', 'PIGTAIL_WINNER'),  # (1) vs pigtail winner
    ('Jett Strickenberger', 'Ezekiel Witt'),  # (17) vs (16)
    ('Maximo Renteria', 'Ayden Smith'),       # (9) vs (24)
    ('Kael Lauridsen', 'Dean Peterson'),      # (25) vs (8)
    ('Troy Spratley', 'Andrew Binni'),        # (5) vs (28)
    ('Conrad Hendriksen', 'Vincent Robinson'), # (21) vs (12)
    ('Stevo Poulin', 'Diego Sotelo'),         # (13) vs (20)
    ('Tyler Chappell', 'Sheldon Seymour'),    # (29) vs (4)
    ('Nic Bouzakis', 'Sulayman Bah'),         # (3) vs (30)
    ('Kysen Terukina', 'Jacob Moran'),        # (19) vs (14)
    ('Tyler Klinsky', 'Davis Motyka'),        # (11) vs (22)
    ('Brady Roark', 'Jore Volk'),             # (27) vs (6)
    ('Nico Provo', 'Cooper Flynn'),           # (7) vs (26)
    ('Nicolar Rivera', 'Marc-Anthony McGowan'), # (23) vs (10)
    ('Koda Holeman', 'Spencer Moore'),        # (15) vs (18)
    ('Desmond Pleasant', 'Eddie Ventresca')   # (31) vs (2)
]

bracket_125.add_round_of_32(round_of_32_125)


# ============================================
# SIMULATE 125lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 125lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_125 = {}

for model in models:
    results_125[model] = bracket_125.simulate(master_pred_df, model, verbose=True)


# ============================================
# SUMMARY TABLE
# ============================================

print("\n" + "="*100)
print("📊 125lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data = []
for model in models:
    res = results_125[model]
    summary_data.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)


# ============================================
# CLEAN HEAD-TO-HEAD FUNCTION
# ============================================

def head_to_head(w1, w2, master_df, weight_class=None):
    """
    Get all model predictions for a matchup.
    Returns dictionary without print statements.
    """
    if weight_class:
        mask = (master_df['weight_class'] == weight_class)
    else:
        mask = pd.Series([True] * len(master_df))

    mask = mask & (
        ((master_df['home_wrestler'] == w1) & (master_df['away_wrestler'] == w2)) |
        ((master_df['home_wrestler'] == w2) & (master_df['away_wrestler'] == w1))
    )

    matchup = master_df[mask]

    if len(matchup) == 0:
        return {'error': f'No matchup found between {w1} and {w2}'}

    row = matchup.iloc[0]
    weight = row['weight_class']

    results = {}
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        results[model] = {
            'winner': row[f'{model}_winner'],
            'probability': round(row[f'{model}_winner_prob'] * 100, 1)
        }

    # Add voting classifier
    voting_winner, voting_prob = voting_classifier(w1, w2, weight, master_df)
    results['voting'] = {
        'winner': voting_winner,
        'probability': round(voting_prob * 100, 1)
    }

    return {
        'weight_class': weight,
        'matchup': f"{w1} vs {w2}",
        'results': results,
        'all_agree': row['all_agree']
    }


🏆 SETTING UP 125lbs BRACKET

🏆 SIMULATING 125lbs BRACKET

🏆 125lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Jace Schafer vs Mack Mauger: Jace Schafer wins (56.1%)

🔹 ROUND OF 32
----------------------------------------
Luke Lilledahl vs Jace Schafer: Luke Lilledahl wins (71.2%)
Jett Strickenberger vs Ezekiel Witt: Jett Strickenberger wins (52.3%)
Maximo Renteria vs Ayden Smith: Maximo Renteria wins (66.0%)
Kael Lauridsen vs Dean Peterson: Dean Peterson wins (71.3%)
Troy Spratley vs Andrew Binni: Troy Spratley wins (61.0%)
Conrad Hendriksen vs Vincent Robinson: Vincent Robinson wins (67.8%)
Stevo Poulin vs Diego Sotelo: Stevo Poulin wins (59.2%)
Tyler Chappell vs Sheldon Seymour: Sheldon Seymour wins (80.4%)
Nic Bouzakis vs Sulayman Bah: Nic Bouzakis wins (61.8%)
Kysen Terukina vs Jacob Moran: Jacob Moran wins (54.9%)
Tyler Klinsky vs Davis Motyka: Tyler Klinsky wins (61.0%)
Brady Roark vs Jore Volk: Brady Roark wins (51.3%)
Nico Provo vs Cooper Flynn: C

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Jace Schafer,"Luke Lilledahl, Sheldon Seymour, Tyler Klinsky...","Luke Lilledahl, Tyler Klinsky",Luke Lilledahl,50.2%
1,DT,Jace Schafer,"Luke Lilledahl, Troy Spratley, Nic Bouzakis, M...","Luke Lilledahl, Nic Bouzakis",Luke Lilledahl,61.1%
2,XGB1,Jace Schafer,"Luke Lilledahl, Sheldon Seymour, Nic Bouzakis,...","Luke Lilledahl, Nic Bouzakis",Luke Lilledahl,61.7%
3,XGB2,Mack Mauger,"Luke Lilledahl, Troy Spratley, Nic Bouzakis, E...","Luke Lilledahl, Nic Bouzakis",Luke Lilledahl,53.2%
4,VOTING,Jace Schafer,"Luke Lilledahl, Troy Spratley, Nic Bouzakis, E...","Luke Lilledahl, Nic Bouzakis",Luke Lilledahl,56.5%


In [87]:
# ============================================
# CREATE 133lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 133lbs BRACKET")
print("="*100)

bracket_133 = Bracket(133, wrestlers_133)

# Pigtail match (33 vs 32)
bracket_133.add_pigtail('Carter Schmidt', 'Andrew Austin')

# Round of 32 matchups (in order)
round_of_32_133 = [
    ('Jax Forrest', 'PIGTAIL_WINNER'),        # (1) vs pigtail winner
    ('TK Davis', 'Zan Fugitt'),                # (17) vs (16)
    ('Dominick Serrano', 'Blake Boarman'),     # (9) vs (24)
    ('Will Betancourt', 'Markel Baker'),       # (25) vs (8)
    ('Kyler Larkin', 'Garrett Grice'),         # (5) vs (28)
    ('Sean Spidle', 'Evan Mougalian'),         # (21) vs (12)
    ('Jacob Van Dee', 'Julian Farber'),        # (13) vs (20)
    ('Luke Willochell', 'Aaron Seidel'),       # (29) vs (4)
    ('Marcus Blaze', 'Gabe Whisenhunt'),       # (3) vs (30)
    ('Gage Walker', 'Ethan Berginc'),          # (19) vs (14)
    ('Tyler Ferrera', 'Zach Redding'),         # (11) vs (22)
    ('Marcel Lopez', 'Drake Ayala'),           # (27) vs (6)
    ('Lucas Byrd', 'Dylan Shawver'),           # (7) vs (26)
    ('Braxton Brown', 'Maximillian Leete'),    # (23) vs (10)
    ('Tyler Knox', 'Gunner Andrick'),          # (15) vs (18)
    ('Gable Strickland', 'Ben Davino')         # (31) vs (2)
]

bracket_133.add_round_of_32(round_of_32_133)

# ============================================
# VERIFY ROUND OF 32 COUNT
# ============================================

print(f"\n📋 133lbs Bracket Setup:")
print(f"   Pigtail matches: {len(bracket_133.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_133.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_133.rounds['pigtail']) + len(bracket_133.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# SIMULATE 133lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 133lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_133 = {}

for model in models:
    results_133[model] = bracket_133.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 133lbs
# ============================================

print("\n" + "="*100)
print("📊 133lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_133 = []
for model in models:
    res = results_133[model]
    summary_data_133.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_133_df = pd.DataFrame(summary_data_133)
display(summary_133_df)

# ============================================
# COMPARE 125lbs vs 133lbs
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs Champion': champ_125,
        '133lbs Champion': champ_133
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC MATCHUPS IN 133lbs
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 133lbs MATCHUPS")
print("="*100)

key_matchups = [
    ('Jax Forrest', 'Ben Davino'),
    ('Marcus Blaze', 'Drake Ayala'),
    ('TK Davis', 'Lucas Byrd'),
    ('Aaron Seidel', 'Ben Davino')
]

for w1, w2 in key_matchups:
    result = head_to_head(w1, w2, master_pred_df, 133)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")


🏆 SETTING UP 133lbs BRACKET

📋 133lbs Bracket Setup:
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🏆 SIMULATING 133lbs BRACKET

🏆 133lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Carter Schmidt vs Andrew Austin: Andrew Austin wins (69.0%)

🔹 ROUND OF 32
----------------------------------------
Jax Forrest vs Andrew Austin: Jax Forrest wins (72.4%)
TK Davis vs Zan Fugitt: TK Davis wins (62.3%)
Dominick Serrano vs Blake Boarman: Dominick Serrano wins (70.7%)
Will Betancourt vs Markel Baker: Markel Baker wins (61.9%)
Kyler Larkin vs Garrett Grice: Kyler Larkin wins (50.2%)
Sean Spidle vs Evan Mougalian: Evan Mougalian wins (54.9%)
Jacob Van Dee vs Julian Farber: Jacob Van Dee wins (57.4%)
Luke Willochell vs Aaron Seidel: Aaron Seidel wins (55.7%)
Marcus Blaze vs Gabe Whisenhunt: Marcus Blaze wins (67.6%)
Gage Walker vs Ethan Berginc: Ethan Berginc wins (61.7%)
Tyler Fe

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Andrew Austin,"Jax Forrest, Kyler Larkin, Marcus Blaze, Lucas...","Jax Forrest, Lucas Byrd",Lucas Byrd,50.2%
1,DT,Andrew Austin,"Jax Forrest, Aaron Seidel, Marcus Blaze, Ben D...","Jax Forrest, Marcus Blaze",Jax Forrest,53.1%
2,XGB1,Andrew Austin,"Jax Forrest, Aaron Seidel, Marcus Blaze, Ben D...","Jax Forrest, Marcus Blaze",Marcus Blaze,59.4%
3,XGB2,Andrew Austin,"Jax Forrest, Jacob Van Dee, Marcus Blaze, Luca...","Jax Forrest, Lucas Byrd",Lucas Byrd,54.0%
4,VOTING,Andrew Austin,"Jax Forrest, Aaron Seidel, Marcus Blaze, Lucas...","Jax Forrest, Lucas Byrd",Lucas Byrd,52.1%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs Champion,133lbs Champion
0,LOGREG,Luke Lilledahl,Lucas Byrd
1,DT,Luke Lilledahl,Jax Forrest
2,XGB1,Luke Lilledahl,Marcus Blaze
3,XGB2,Luke Lilledahl,Lucas Byrd
4,VOTING,Luke Lilledahl,Lucas Byrd



🔍 TESTING KEY 133lbs MATCHUPS

📊 Jax Forrest vs Ben Davino
--------------------------------------------------
LOGREG  : Jax Forrest (52.6%)
DT      : Jax Forrest (53.1%)
XGB1    : Ben Davino (54.099998474121094%)
XGB2    : Ben Davino (66.4000015258789%)
VOTING  : Jax Forrest (52.9%)

📊 Marcus Blaze vs Drake Ayala
--------------------------------------------------
LOGREG  : Marcus Blaze (72.4%)
DT      : Marcus Blaze (74.6%)
XGB1    : Marcus Blaze (73.5%)
XGB2    : Marcus Blaze (64.5999984741211%)
VOTING  : Marcus Blaze (70.5%)

📊 TK Davis vs Lucas Byrd
--------------------------------------------------
LOGREG  : Lucas Byrd (50.2%)
DT      : TK Davis (53.1%)
XGB1    : TK Davis (54.900001525878906%)
XGB2    : Lucas Byrd (81.80000305175781%)
VOTING  : Lucas Byrd (66.0%)

📊 Aaron Seidel vs Ben Davino
--------------------------------------------------
LOGREG  : Ben Davino (58.2%)
DT      : Aaron Seidel (53.1%)
XGB1    : Ben Davino (61.599998474121094%)
XGB2    : Ben Davino (75.199996948242

In [89]:
# ============================================
# CREATE 141lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 141lbs BRACKET")
print("="*100)

# First, let's define the wrestlers list if not already defined


bracket_141 = Bracket(141, wrestlers_141)

# Pigtail match (33 vs 32)
bracket_141.add_pigtail('Aldo Hernandez', 'Matthew Martino')

# Round of 32 matchups
round_of_32_141 = [
    ('Jesse Mendez', 'PIGTAIL_WINNER'),        # (1) vs pigtail winner
    ('Caedyn Ricciardi', 'Ryan Jack'),          # (17) vs (16)
    ('Joey Olivieri', 'Nash Singleton'),        # (9) vs (24)
    ('Tom Crook', 'Vance Vombaur'),             # (25) vs (8)
    ('Luke Stanich', 'Pierson Manville'),       # (5) vs (28)
    ('Tyler Wells', 'Luke Simcox'),             # (21) vs (12)
    ('Wyatt Henson', 'Julian Tagg'),            # (13) vs (20)
    ('Jordan Titus', 'Anthony Echemendia'),     # (29) vs (4)
    ('Brock Hardy', 'Dario Lemus'),             # (3) vs (30)
    ('Haiden Drury', 'Braeden Davis'),          # (19) vs (14)
    ('CJ Composto', 'Lorenzo Frezza'),          # (11) vs (22)
    ('Gable Porter', 'Vince Cornella'),         # (27) vs (6)
    ('Nasir Bailey', 'Braden Basile'),          # (7) vs (26)
    ('Dylan Chappell', 'Jack Consiglio'),       # (23) vs (10)
    ('Eli Griffin', 'Carter Nogle'),         # (15) vs (18)
    ('Billy Dekraker', 'Sergio Vega')           # (31) vs (2)
]

bracket_141.add_round_of_32(round_of_32_141)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 141lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_141)}")
print(f"   Pigtail matches: {len(bracket_141.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_141.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_141.rounds['pigtail']) + len(bracket_141.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 141lbs matchups exist in predictions:")

key_matchups = [
    ('Jesse Mendez', 'Anthony Echemendia'),
    ('Brock Hardy', 'Vince Cornella'),
    ('CJ Composto', 'Lorenzo Frezza'),
    ('Sergio Vega', 'Billy Dekraker'),
    ('Anthony Echemendia', 'Jordan Titus')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 141lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 141lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_141 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_141[model] = bracket_141.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 141lbs
# ============================================

print("\n" + "="*100)
print("📊 141lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_141 = []
for model in models:
    res = results_141[model]
    summary_data_141.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_141_df = pd.DataFrame(summary_data_141)
display(summary_141_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 141lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 141lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Jesse Mendez', 'Anthony Echemendia'),
    ('Brock Hardy', 'Vince Cornella'),
    ('CJ Composto', 'Lorenzo Frezza'),
    ('Sergio Vega', 'Billy Dekraker')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 141)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 141lbs BRACKET

📋 141lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 141lbs matchups exist in predictions:
✅ Jesse Mendez vs Anthony Echemendia: FOUND
✅ Brock Hardy vs Vince Cornella: FOUND
✅ CJ Composto vs Lorenzo Frezza: FOUND
✅ Sergio Vega vs Billy Dekraker: FOUND
✅ Anthony Echemendia vs Jordan Titus: FOUND

🏆 SIMULATING 141lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 141lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Aldo Hernandez vs Matthew Martino: Aldo Hernandez wins (53.9%)

🔹 ROUND OF 32
----------------------------------------
Jesse Mendez vs Aldo Hernandez: Jesse Mendez wins (76.5%)
Caedyn Ricciardi vs Ryan Jack: Caedyn Ricciardi wins (56.9%)
Joey Olivieri vs Nash Singleton: Joey Olivieri wins (69.6%)
Tom Crook vs Vance Vombaur: Vance Vombaur wins (66.8%)
Luke Stanich vs Pierson Manville: Lu

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Aldo Hernandez,"Jesse Mendez, Luke Stanich, Vince Cornella, Se...","Jesse Mendez, Sergio Vega",Jesse Mendez,50.2%
1,DT,Aldo Hernandez,"Jesse Mendez, Luke Stanich, Brock Hardy, Sergi...","Jesse Mendez, Sergio Vega",Jesse Mendez,61.1%
2,XGB1,Aldo Hernandez,"Jesse Mendez, Wyatt Henson, Brock Hardy, Sergi...","Jesse Mendez, Brock Hardy",Jesse Mendez,75.1%
3,XGB2,Matthew Martino,"Jesse Mendez, Anthony Echemendia, Brock Hardy,...","Jesse Mendez, Brock Hardy",Jesse Mendez,65.7%
4,VOTING,Aldo Hernandez,"Jesse Mendez, Luke Stanich, Brock Hardy, Sergi...","Jesse Mendez, Sergio Vega",Jesse Mendez,61.9%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez



🔍 TESTING KEY 141lbs MATCHUPS

📊 Jesse Mendez vs Anthony Echemendia
--------------------------------------------------
LOGREG  : Jesse Mendez (60.0%)
DT      : Jesse Mendez (74.6%)
XGB1    : Jesse Mendez (73.0999984741211%)
XGB2    : Jesse Mendez (79.5999984741211%)
VOTING  : Jesse Mendez (71.4%)
All Agree: ✅

📊 Brock Hardy vs Vince Cornella
--------------------------------------------------
LOGREG  : Vince Cornella (57.6%)
DT      : Brock Hardy (61.1%)
XGB1    : Brock Hardy (63.70000076293945%)
XGB2    : Brock Hardy (74.4000015258789%)
VOTING  : Brock Hardy (67.7%)
All Agree: ❌

📊 CJ Composto vs Lorenzo Frezza
--------------------------------------------------
LOGREG  : CJ Composto (57.6%)
DT      : CJ Composto (61.1%)
XGB1    : CJ Composto (55.79999923706055%)
XGB2    : CJ Composto (61.400001525878906%)
VOTING  : CJ Composto (60.0%)
All Agree: ✅

📊 Sergio Vega vs Billy Dekraker
--------------------------------------------------
LOGREG  : Sergio Vega (75.0%)
DT      : Sergio Vega (89

In [90]:
# ============================================
# CREATE 149lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 149lbs BRACKET")
print("="*100)


bracket_149 = Bracket(149, wrestlers_149)

# Pigtail match (33 vs 32)
bracket_149.add_pigtail('Austin McBurney', 'Clayton Jones')

# Round of 32 matchups
round_of_32_149 = [
    ('Shayne Van Ness', 'PIGTAIL_WINNER'),     # (1) vs pigtail winner
    ('Lucas Kapusta', 'Jacob Frost'),           # (17) vs (16)
    ('David Evans', 'Andrew Clark'),            # (9) vs (24)
    ('Michael Gioffre', 'Casey Swiderski'),     # (25) vs (8)
    ('Koy Buesgens', 'Kade Brown'),             # (5) vs (28)
    ('Gabe Willochell', 'Carter Young'),        # (21) vs (12)
    ('Joseph Zargo', 'Chance Lamer'),           # (13) vs (20)
    ('Kaden Cassidy', 'Collin Gaj'),            # (29) vs (4)
    ('Cross Wasilewski', 'Dylan Layton'),       # (3) vs (30)
    ('Brock Herman', 'Caleb Rathjen'),          # (19) vs (14)
    ('Lachlan McNeil', 'Eligh Rivera'),         # (11) vs (22)
    ('Andre Gonzales', 'Caleb Tyus'),           # (27) vs (6)
    ('Ethan Stiles', 'Anderson Heap'),          # (7) vs (26)
    ('Maxwell Petersen', 'Aden Valencia'),      # (23) vs (10)
    ('Ryder Block', 'Eugene Harney'),           # (15) vs (18)
    ('Ryan Michaels', 'Jaxon Joy')              # (31) vs (2)
]

bracket_149.add_round_of_32(round_of_32_149)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 149lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_149)}")
print(f"   Pigtail matches: {len(bracket_149.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_149.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_149.rounds['pigtail']) + len(bracket_149.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 149lbs matchups exist in predictions:")

key_matchups = [
    ('Shayne Van Ness', 'Jaxon Joy'),
    ('Cross Wasilewski', 'Collin Gaj'),
    ('Koy Buesgens', 'Caleb Tyus'),
    ('Ethan Stiles', 'Aden Valencia'),
    ('Lachlan McNeil', 'Carter Young')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 149lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 149lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_149 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_149[model] = bracket_149.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 149lbs
# ============================================

print("\n" + "="*100)
print("📊 149lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_149 = []
for model in models:
    res = results_149[model]
    summary_data_149.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_149_df = pd.DataFrame(summary_data_149)
display(summary_149_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion'] if 'results_141' in globals() else 'N/A'
    champ_149 = results_149[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141,
        '149lbs': champ_149
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 149lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 149lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Shayne Van Ness', 'Jaxon Joy'),
    ('Cross Wasilewski', 'Collin Gaj'),
    ('Koy Buesgens', 'Caleb Tyus'),
    ('Ethan Stiles', 'Aden Valencia'),
    ('Lachlan McNeil', 'Carter Young')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 149)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 149lbs BRACKET

📋 149lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 149lbs matchups exist in predictions:
✅ Shayne Van Ness vs Jaxon Joy: FOUND
✅ Cross Wasilewski vs Collin Gaj: FOUND
✅ Koy Buesgens vs Caleb Tyus: FOUND
✅ Ethan Stiles vs Aden Valencia: FOUND
✅ Lachlan McNeil vs Carter Young: FOUND

🏆 SIMULATING 149lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 149lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Austin McBurney vs Clayton Jones: Austin McBurney wins (60.3%)

🔹 ROUND OF 32
----------------------------------------
Shayne Van Ness vs Austin McBurney: Shayne Van Ness wins (72.0%)
Lucas Kapusta vs Jacob Frost: Lucas Kapusta wins (56.1%)
David Evans vs Andrew Clark: David Evans wins (61.6%)
Michael Gioffre vs Casey Swiderski: Casey Swiderski wins (57.6%)
Koy Buesgens vs Kade Brown: Koy Buesgens 

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Austin McBurney,"Shayne Van Ness, Koy Buesgens, Caleb Tyus, Jax...","Shayne Van Ness, Caleb Tyus",Shayne Van Ness,50.2%
1,DT,Austin McBurney,"Shayne Van Ness, Kaden Cassidy, Cross Wasilews...","Shayne Van Ness, Jaxon Joy",Shayne Van Ness,61.1%
2,XGB1,Austin McBurney,"Shayne Van Ness, Koy Buesgens, Cross Wasilewsk...","Shayne Van Ness, Jaxon Joy",Shayne Van Ness,67.8%
3,XGB2,Austin McBurney,"Shayne Van Ness, Koy Buesgens, Lachlan McNeil,...","Shayne Van Ness, Lachlan McNeil",Shayne Van Ness,77.3%
4,VOTING,Austin McBurney,"Shayne Van Ness, Koy Buesgens, Cross Wasilewsk...","Shayne Van Ness, Jaxon Joy",Shayne Van Ness,60.9%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs,149lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness



🔍 TESTING KEY 149lbs MATCHUPS

📊 Shayne Van Ness vs Jaxon Joy
--------------------------------------------------
LOGREG  : Shayne Van Ness (53.9%)
DT      : Shayne Van Ness (61.1%)
XGB1    : Shayne Van Ness (67.80000305175781%)
XGB2    : Shayne Van Ness (67.69999694824219%)
VOTING  : Shayne Van Ness (60.9%)
All Agree: ✅

📊 Cross Wasilewski vs Collin Gaj
--------------------------------------------------
LOGREG  : Cross Wasilewski (72.8%)
DT      : Cross Wasilewski (85.6%)
XGB1    : Cross Wasilewski (87.0%)
XGB2    : Cross Wasilewski (67.30000305175781%)
VOTING  : Cross Wasilewski (75.2%)
All Agree: ✅

📊 Koy Buesgens vs Caleb Tyus
--------------------------------------------------
LOGREG  : Caleb Tyus (56.1%)
DT      : Caleb Tyus (53.1%)
XGB1    : Caleb Tyus (50.70000076293945%)
XGB2    : Koy Buesgens (85.80000305175781%)
VOTING  : Caleb Tyus (54.6%)
All Agree: ❌

📊 Ethan Stiles vs Aden Valencia
--------------------------------------------------
LOGREG  : Ethan Stiles (59.2%)
DT      :

In [92]:
# ============================================
# CREATE 157lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 157lbs BRACKET")
print("="*100)


bracket_157 = Bracket(157, wrestlers_157)

# Pigtail match (33 vs 32)
bracket_157.add_pigtail('Yannis Charles', 'Jeb Prechtel')

# Round of 32 matchups
round_of_32_157 = [
    ('PJ Duke', 'PIGTAIL_WINNER'),             # (1) vs pigtail winner
    ('Luke Mechler', 'Cael Swensen'),           # (17) vs (16)
    ('Daniel Cardenas', 'Jaivon Jones'),        # (9) vs (24)
    ('Mason Shrader', 'Brandon Cannon'),        # (25) vs (8)
    ('Landon Robideau', 'Gavin Drexler'),       # (5) vs (28)
    ('Charlie Millard', 'Vince Zerban'),        # (21) vs (12)
    ('Derek Raike', 'Jimmy Harrington'),        # (13) vs (20)
    ('Bryce Lowery', 'Kaleb Larkin'),           # (29) vs (4)
    ('Meyer Shapiro', 'Laird Root'),            # (3) vs (30)
    ('Kai Owen', 'Ethen Miller'),               # (19) vs (14)
    ('Ty Watters', 'Colton Washleski'),         # (11) vs (22)
    ('Dylan Evans', 'Jude Swisher'),            # (27) vs (6)
    ('Kannon Webster', 'Jonathan Ley'),         # (7) vs (26)
    ('Kaleb Burgess', 'Logan Rozynski'),        # (23) vs (10)
    ('Cam Catrabone', 'DJ McGee'),              # (15) vs (18)
    ('Garrett McChesney', 'Antrell Taylor')     # (31) vs (2)
]

bracket_157.add_round_of_32(round_of_32_157)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 157lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_157)}")
print(f"   Pigtail matches: {len(bracket_157.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_157.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_157.rounds['pigtail']) + len(bracket_157.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 157lbs matchups exist in predictions:")

key_matchups = [
    ('PJ Duke', 'Antrell Taylor'),
    ('Meyer Shapiro', 'Jude Swisher'),
    ('Daniel Cardenas', 'Brandon Cannon'),
    ('Kannon Webster', 'Ty Watters'),
    ('Cam Catrabone', 'DJ McGee')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 157lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 157lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_157 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_157[model] = bracket_157.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 157lbs
# ============================================

print("\n" + "="*100)
print("📊 157lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_157 = []
for model in models:
    res = results_157[model]
    summary_data_157.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_157_df = pd.DataFrame(summary_data_157)
display(summary_157_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion'] if 'results_141' in globals() else 'N/A'
    champ_149 = results_149[model]['champion'] if 'results_149' in globals() else 'N/A'
    champ_157 = results_157[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141,
        '149lbs': champ_149,
        '157lbs': champ_157
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 157lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 157lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('PJ Duke', 'Antrell Taylor'),
    ('Meyer Shapiro', 'Jude Swisher'),
    ('Daniel Cardenas', 'Brandon Cannon'),
    ('Kannon Webster', 'Ty Watters'),
    ('Cam Catrabone', 'DJ McGee')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 157)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 157lbs BRACKET

📋 157lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 157lbs matchups exist in predictions:
✅ PJ Duke vs Antrell Taylor: FOUND
✅ Meyer Shapiro vs Jude Swisher: FOUND
✅ Daniel Cardenas vs Brandon Cannon: FOUND
✅ Kannon Webster vs Ty Watters: FOUND
✅ Cam Catrabone vs DJ McGee: FOUND

🏆 SIMULATING 157lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 157lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Yannis Charles vs Jeb Prechtel: Jeb Prechtel wins (63.9%)

🔹 ROUND OF 32
----------------------------------------
PJ Duke vs Jeb Prechtel: PJ Duke wins (75.9%)
Luke Mechler vs Cael Swensen: Cael Swensen wins (59.3%)
Daniel Cardenas vs Jaivon Jones: Daniel Cardenas wins (62.7%)
Mason Shrader vs Brandon Cannon: Brandon Cannon wins (61.0%)
Landon Robideau vs Gavin Drexler: Landon Robideau wins (67.6%)
Ch

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Jeb Prechtel,"Brandon Cannon, Landon Robideau, Meyer Shapiro...","Brandon Cannon, Meyer Shapiro",Meyer Shapiro,50.2%
1,DT,Yannis Charles,"Brandon Cannon, Landon Robideau, Meyer Shapiro...","Brandon Cannon, Meyer Shapiro",Meyer Shapiro,61.1%
2,XGB1,Jeb Prechtel,"Brandon Cannon, Landon Robideau, Meyer Shapiro...","Brandon Cannon, Meyer Shapiro",Meyer Shapiro,58.1%
3,XGB2,Jeb Prechtel,"Brandon Cannon, Landon Robideau, Meyer Shapiro...","Brandon Cannon, Antrell Taylor",Brandon Cannon,62.1%
4,VOTING,Jeb Prechtel,"Brandon Cannon, Landon Robideau, Meyer Shapiro...","Brandon Cannon, Meyer Shapiro",Meyer Shapiro,55.7%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs,149lbs,157lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness,Meyer Shapiro
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness,Meyer Shapiro
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Brandon Cannon
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro



🔍 TESTING KEY 157lbs MATCHUPS

📊 PJ Duke vs Antrell Taylor
--------------------------------------------------
LOGREG  : PJ Duke (52.0%)
DT      : PJ Duke (53.1%)
XGB1    : Antrell Taylor (57.599998474121094%)
XGB2    : PJ Duke (50.20000076293945%)
VOTING  : PJ Duke (51.7%)
All Agree: ❌

📊 Meyer Shapiro vs Jude Swisher
--------------------------------------------------
LOGREG  : Meyer Shapiro (63.5%)
DT      : Meyer Shapiro (74.6%)
XGB1    : Meyer Shapiro (78.0%)
XGB2    : Meyer Shapiro (86.5999984741211%)
VOTING  : Meyer Shapiro (74.9%)
All Agree: ✅

📊 Daniel Cardenas vs Brandon Cannon
--------------------------------------------------
LOGREG  : Brandon Cannon (59.2%)
DT      : Brandon Cannon (61.1%)
XGB1    : Brandon Cannon (62.70000076293945%)
XGB2    : Brandon Cannon (72.80000305175781%)
VOTING  : Brandon Cannon (64.4%)
All Agree: ✅

📊 Kannon Webster vs Ty Watters
--------------------------------------------------
LOGREG  : Kannon Webster (50.2%)
DT      : Ty Watters (53.1%)
XGB1  

In [93]:
# ============================================
# CREATE 165lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 165lbs BRACKET")
print("="*100)


bracket_165 = Bracket(165, wrestlers_165)

# Pigtail match (33 vs 32)
bracket_165.add_pigtail('Ryan Vigil', 'Cody Walsh')

# Round of 32 matchups
round_of_32_165 = [
    ('Mitchell Mesenbrink', 'PIGTAIL_WINNER'),   # (1) vs pigtail winner
    ('Braeden Scoles', 'Paddy Gallagher'),       # (17) vs (16)
    ('Bryce Hepner', 'Sean Seefeldt'),           # (9) vs (24)
    ('Mac Church', 'Matt Bianchi'),              # (25) vs (8)
    ('Ladarion Lockett', 'Cody Goebel'),         # (5) vs (28)
    ('Brock Woodcock', 'Cesar Alvan'),           # (21) vs (12)
    ('Andrew Sparks', 'Ty Whalen'),              # (13) vs (20)
    ('Ryan Burgos', 'Nicco Ruiz'),               # (29) vs (4)
    ('Michael Caliendo', 'Thomas Snipes'),       # (3) vs (30)
    ('Noah Mulvaney', 'Andrew Barbosa'),         # (19) vs (14)
    ('Ryder Downey', 'Matthew Olguin'),          # (11) vs (22)
    ('EJ Parco', 'LJ Araujo'),                   # (27) vs (6)
    ('Max Brignola', 'Tyler Lillard'),           # (7) vs (26)
    ('Chris Earnest', 'Will Denny'),             # (23) vs (10)
    ('Connor Euton', 'Gunner Filipowicz'),       # (15) vs (18)
    ('Jared Keslar', 'Joey Blaze')               # (31) vs (2)
]

bracket_165.add_round_of_32(round_of_32_165)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 165lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_165)}")
print(f"   Pigtail matches: {len(bracket_165.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_165.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_165.rounds['pigtail']) + len(bracket_165.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 165lbs matchups exist in predictions:")

key_matchups = [
    ('Mitchell Mesenbrink', 'Joey Blaze'),
    ('Michael Caliendo', 'LJ Araujo'),
    ('Cesar Alvan', 'Will Denny'),
    ('Bryce Hepner', 'Andrew Sparks'),
    ('Max Brignola', 'Tyler Lillard')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 165lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 165lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_165 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_165[model] = bracket_165.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 165lbs
# ============================================

print("\n" + "="*100)
print("📊 165lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_165 = []
for model in models:
    res = results_165[model]
    summary_data_165.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_165_df = pd.DataFrame(summary_data_165)
display(summary_165_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion'] if 'results_141' in globals() else 'N/A'
    champ_149 = results_149[model]['champion'] if 'results_149' in globals() else 'N/A'
    champ_157 = results_157[model]['champion'] if 'results_157' in globals() else 'N/A'
    champ_165 = results_165[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141,
        '149lbs': champ_149,
        '157lbs': champ_157,
        '165lbs': champ_165
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 165lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 165lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Mitchell Mesenbrink', 'Joey Blaze'),
    ('Michael Caliendo', 'LJ Araujo'),
    ('Cesar Alvan', 'Will Denny'),
    ('Bryce Hepner', 'Andrew Sparks'),
    ('Max Brignola', 'Tyler Lillard')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 165)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 165lbs BRACKET

📋 165lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 165lbs matchups exist in predictions:
✅ Mitchell Mesenbrink vs Joey Blaze: FOUND
✅ Michael Caliendo vs LJ Araujo: FOUND
✅ Cesar Alvan vs Will Denny: FOUND
✅ Bryce Hepner vs Andrew Sparks: FOUND
✅ Max Brignola vs Tyler Lillard: FOUND

🏆 SIMULATING 165lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 165lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Ryan Vigil vs Cody Walsh: Cody Walsh wins (64.9%)

🔹 ROUND OF 32
----------------------------------------
Mitchell Mesenbrink vs Cody Walsh: Mitchell Mesenbrink wins (64.9%)
Braeden Scoles vs Paddy Gallagher: Braeden Scoles wins (58.3%)
Bryce Hepner vs Sean Seefeldt: Sean Seefeldt wins (53.7%)
Mac Church vs Matt Bianchi: Matt Bianchi wins (68.5%)
Ladarion Lockett vs Cody Goebel: Ladarion Lockett 

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Cody Walsh,"Mitchell Mesenbrink, Ladarion Lockett, Noah Mu...","Mitchell Mesenbrink, Joey Blaze",Mitchell Mesenbrink,50.2%
1,DT,Cody Walsh,"Mitchell Mesenbrink, Brock Woodcock, Michael C...","Mitchell Mesenbrink, Joey Blaze",Mitchell Mesenbrink,61.1%
2,XGB1,Cody Walsh,"Mitchell Mesenbrink, Brock Woodcock, Michael C...","Mitchell Mesenbrink, Michael Caliendo",Mitchell Mesenbrink,65.1%
3,XGB2,Cody Walsh,"Mitchell Mesenbrink, Ladarion Lockett, Michael...","Mitchell Mesenbrink, Michael Caliendo",Mitchell Mesenbrink,53.9%
4,VOTING,Cody Walsh,"Mitchell Mesenbrink, Ladarion Lockett, Michael...","Mitchell Mesenbrink, Joey Blaze",Mitchell Mesenbrink,66.9%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs,149lbs,157lbs,165lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Brandon Cannon,Mitchell Mesenbrink
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink



🔍 TESTING KEY 165lbs MATCHUPS

📊 Mitchell Mesenbrink vs Joey Blaze
--------------------------------------------------
LOGREG  : Mitchell Mesenbrink (50.2%)
DT      : Mitchell Mesenbrink (61.1%)
XGB1    : Mitchell Mesenbrink (67.9000015258789%)
XGB2    : Mitchell Mesenbrink (89.30000305175781%)
VOTING  : Mitchell Mesenbrink (66.9%)
All Agree: ✅

📊 Michael Caliendo vs LJ Araujo
--------------------------------------------------
LOGREG  : Michael Caliendo (61.4%)
DT      : Michael Caliendo (89.0%)
XGB1    : Michael Caliendo (82.19999694824219%)
XGB2    : Michael Caliendo (85.0999984741211%)
VOTING  : Michael Caliendo (78.5%)
All Agree: ✅

📊 Cesar Alvan vs Will Denny
--------------------------------------------------
LOGREG  : Will Denny (56.4%)
DT      : Will Denny (61.1%)
XGB1    : Will Denny (67.80000305175781%)
XGB2    : Will Denny (64.69999694824219%)
VOTING  : Will Denny (60.7%)
All Agree: ✅

📊 Bryce Hepner vs Andrew Sparks
--------------------------------------------------
LOGREG  

In [94]:
# ============================================
# CREATE 174lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 174lbs BRACKET")
print("="*100)


bracket_174 = Bracket(174, wrestlers_174)

# Pigtail match (33 vs 32)
bracket_174.add_pigtail("Grant O'Dell", 'Lucas Condon')

# Round of 32 matchups
round_of_32_174 = [
    ('Levi Haines', 'PIGTAIL_WINNER'),          # (1) vs pigtail winner
    ('Jared Simma', 'Nick Fine'),                # (17) vs (16)
    ('Beau Mantanona', 'Garrett Thompson'),      # (9) vs (24)
    ('Sergio Desiante', 'Alex Facundo'),         # (25) vs (8)
    ('Patrick Kennedy', 'Holden Garcia'),        # (5) vs (28)
    ('Lenny Pinto', 'Carter Schubert'),          # (21) vs (12)
    ('Carter Baer', 'Daschle Lamer'),            # (13) vs (20)
    ('Avery Bassett', 'Carson Kharchla'),        # (29) vs (4)
    ('Christopher Minto', 'Riley Davis'),        # (3) vs (30)
    ('Logan Messer', 'Moses Espinoza-Owens'),    # (19) vs (14)
    ('MJ Gaitan', 'Brody Baumann'),              # (11) vs (22)
    ('Collin Carrigan', 'Matthew Singleton'),    # (27) vs (6)
    ('Cameron Steed', 'Derek Gilcher'),          # (7) vs (26)
    ('Luca Augustine', 'Myles Takats'),          # (23) vs (10)
    ('Danny Wask', 'Colin Kelly'),               # (15) vs (18)
    ('Cael Valencia', 'Simon Ruiz')              # (31) vs (2)
]

bracket_174.add_round_of_32(round_of_32_174)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 174lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_174)}")
print(f"   Pigtail matches: {len(bracket_174.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_174.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_174.rounds['pigtail']) + len(bracket_174.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 174lbs matchups exist in predictions:")

key_matchups = [
    ('Levi Haines', 'Simon Ruiz'),
    ('Christopher Minto', 'Carson Kharchla'),
    ('Patrick Kennedy', 'Matthew Singleton'),
    ('Alex Facundo', 'Cameron Steed'),
    ('Beau Mantanona', 'Myles Takats')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 174lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 174lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_174 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_174[model] = bracket_174.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 174lbs
# ============================================

print("\n" + "="*100)
print("📊 174lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_174 = []
for model in models:
    res = results_174[model]
    summary_data_174.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_174_df = pd.DataFrame(summary_data_174)
display(summary_174_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion'] if 'results_141' in globals() else 'N/A'
    champ_149 = results_149[model]['champion'] if 'results_149' in globals() else 'N/A'
    champ_157 = results_157[model]['champion'] if 'results_157' in globals() else 'N/A'
    champ_165 = results_165[model]['champion'] if 'results_165' in globals() else 'N/A'
    champ_174 = results_174[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141,
        '149lbs': champ_149,
        '157lbs': champ_157,
        '165lbs': champ_165,
        '174lbs': champ_174
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 174lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 174lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Levi Haines', 'Simon Ruiz'),
    ('Christopher Minto', 'Carson Kharchla'),
    ('Patrick Kennedy', 'Matthew Singleton'),
    ('Alex Facundo', 'Cameron Steed'),
    ('Beau Mantanona', 'Myles Takats')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 174)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 174lbs BRACKET

📋 174lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 174lbs matchups exist in predictions:
✅ Levi Haines vs Simon Ruiz: FOUND
✅ Christopher Minto vs Carson Kharchla: FOUND
✅ Patrick Kennedy vs Matthew Singleton: FOUND
✅ Alex Facundo vs Cameron Steed: FOUND
✅ Beau Mantanona vs Myles Takats: FOUND

🏆 SIMULATING 174lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 174lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Grant O'Dell vs Lucas Condon: Grant O'Dell wins (64.6%)

🔹 ROUND OF 32
----------------------------------------
Levi Haines vs Grant O'Dell: Levi Haines wins (66.4%)
Jared Simma vs Nick Fine: Nick Fine wins (57.5%)
Beau Mantanona vs Garrett Thompson: Garrett Thompson wins (52.2%)
Sergio Desiante vs Alex Facundo: Alex Facundo wins (69.8%)
Patrick Kennedy vs Holden Garcia: Patrick Kenned

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Grant O'Dell,"Levi Haines, Carter Baer, Moses Espinoza-Owens...","Levi Haines, Simon Ruiz",Levi Haines,50.2%
1,DT,Grant O'Dell,"Levi Haines, Carson Kharchla, Christopher Mint...","Levi Haines, Simon Ruiz",Levi Haines,61.1%
2,XGB1,Grant O'Dell,"Levi Haines, Carson Kharchla, Christopher Mint...","Levi Haines, Simon Ruiz",Levi Haines,75.7%
3,XGB2,Lucas Condon,"Levi Haines, Carson Kharchla, Christopher Mint...","Levi Haines, Christopher Minto",Levi Haines,72.0%
4,VOTING,Grant O'Dell,"Levi Haines, Carson Kharchla, Christopher Mint...","Levi Haines, Simon Ruiz",Levi Haines,63.8%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs,149lbs,157lbs,165lbs,174lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Brandon Cannon,Mitchell Mesenbrink,Levi Haines
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines



🔍 TESTING KEY 174lbs MATCHUPS

📊 Levi Haines vs Simon Ruiz
--------------------------------------------------
LOGREG  : Levi Haines (50.2%)
DT      : Levi Haines (61.1%)
XGB1    : Levi Haines (75.69999694824219%)
XGB2    : Levi Haines (80.0%)
VOTING  : Levi Haines (63.8%)
All Agree: ✅

📊 Christopher Minto vs Carson Kharchla
--------------------------------------------------
LOGREG  : Carson Kharchla (51.6%)
DT      : Carson Kharchla (53.1%)
XGB1    : Carson Kharchla (56.900001525878906%)
XGB2    : Carson Kharchla (63.79999923706055%)
VOTING  : Carson Kharchla (56.2%)
All Agree: ✅

📊 Patrick Kennedy vs Matthew Singleton
--------------------------------------------------
LOGREG  : Patrick Kennedy (53.7%)
DT      : Patrick Kennedy (61.1%)
XGB1    : Patrick Kennedy (58.70000076293945%)
XGB2    : Patrick Kennedy (59.29999923706055%)
VOTING  : Patrick Kennedy (58.0%)
All Agree: ✅

📊 Alex Facundo vs Cameron Steed
--------------------------------------------------
LOGREG  : Cameron Steed (58.

In [95]:
# ============================================
# CREATE 184lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 184lbs BRACKET")
print("="*100)


bracket_184 = Bracket(184, wrestlers_184)

# Pigtail match (33 vs 32)
bracket_184.add_pigtail('Sam Goin', 'Caleb Uhlenhopp')

# Round of 32 matchups
round_of_32_184 = [
    ('Rocco Welsh', 'PIGTAIL_WINNER'),          # (1) vs pigtail winner
    ('Ian Bush', 'Rylan Rogers'),                # (17) vs (16)
    ('Chris Moore', 'Joe Curtis'),               # (9) vs (24)
    ('Malachi Duvall', 'Silas Allred'),          # (25) vs (8)
    ('Brock Mantanona', 'Abe Wojcikiewicz'),     # (5) vs (28)
    ('Tomas Brooker', 'Dylan Fishback'),         # (21) vs (12)
    ('Isaac Dean', 'Brian Soldano'),             # (13) vs (20)
    ('Nick Fox', 'James Conway'),                # (29) vs (4)
    ('Max McEnelly', 'Tyler Bienus'),            # (3) vs (30)
    ('Jared McGill', 'Jaden Bullock'),           # (19) vs (14)
    ('Shane Cartagena-Walsh', 'Zack Ryder'),     # (11) vs (22)
    ('Aidan Brenot', 'Eddie Neitenbach'),        # (27) vs (6)
    ('Angelo Ferrari', 'Chase Kranitz'),         # (7) vs (26)
    ('Ceasar Garza', 'Caleb Campos'),            # (23) vs (10)
    ('Sal Perrine', 'Jake Dailey'),              # (15) vs (18)
    ('Mahonri Rushton', 'Aeoden Sinclair')       # (31) vs (2)
]

bracket_184.add_round_of_32(round_of_32_184)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 184lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_184)}")
print(f"   Pigtail matches: {len(bracket_184.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_184.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_184.rounds['pigtail']) + len(bracket_184.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 184lbs matchups exist in predictions:")

key_matchups = [
    ('Rocco Welsh', 'Aeoden Sinclair'),
    ('Max McEnelly', 'James Conway'),
    ('Joe Curtis', 'Silas Allred'),
    ('Jaden Bullock', 'Dylan Fishback'),
    ('Angelo Ferrari', 'Eddie Neitenbach')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 184lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 184lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_184 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_184[model] = bracket_184.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 184lbs
# ============================================

print("\n" + "="*100)
print("📊 184lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_184 = []
for model in models:
    res = results_184[model]
    summary_data_184.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_184_df = pd.DataFrame(summary_data_184)
display(summary_184_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion'] if 'results_141' in globals() else 'N/A'
    champ_149 = results_149[model]['champion'] if 'results_149' in globals() else 'N/A'
    champ_157 = results_157[model]['champion'] if 'results_157' in globals() else 'N/A'
    champ_165 = results_165[model]['champion'] if 'results_165' in globals() else 'N/A'
    champ_174 = results_174[model]['champion'] if 'results_174' in globals() else 'N/A'
    champ_184 = results_184[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141,
        '149lbs': champ_149,
        '157lbs': champ_157,
        '165lbs': champ_165,
        '174lbs': champ_174,
        '184lbs': champ_184
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 184lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 184lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Rocco Welsh', 'Aeoden Sinclair'),
    ('Max McEnelly', 'James Conway'),
    ('Joe Curtis', 'Silas Allred'),
    ('Jaden Bullock', 'Dylan Fishback'),
    ('Angelo Ferrari', 'Eddie Neitenbach')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 184)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 184lbs BRACKET

📋 184lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 184lbs matchups exist in predictions:
✅ Rocco Welsh vs Aeoden Sinclair: FOUND
✅ Max McEnelly vs James Conway: FOUND
✅ Joe Curtis vs Silas Allred: FOUND
✅ Jaden Bullock vs Dylan Fishback: FOUND
✅ Angelo Ferrari vs Eddie Neitenbach: FOUND

🏆 SIMULATING 184lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 184lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Sam Goin vs Caleb Uhlenhopp: Sam Goin wins (54.4%)

🔹 ROUND OF 32
----------------------------------------
Rocco Welsh vs Sam Goin: Rocco Welsh wins (70.0%)
Ian Bush vs Rylan Rogers: Rylan Rogers wins (54.5%)
Chris Moore vs Joe Curtis: Joe Curtis wins (69.1%)
Malachi Duvall vs Silas Allred: Malachi Duvall wins (63.6%)
Brock Mantanona vs Abe Wojcikiewicz: Brock Mantanona wins (60.7%)
Tomas Bro

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Sam Goin,"Rocco Welsh, Tomas Brooker, Max McEnelly, Ceas...","Rocco Welsh, Ceasar Garza",Rocco Welsh,50.2%
1,DT,Sam Goin,"Joe Curtis, Brock Mantanona, Max McEnelly, Aeo...","Joe Curtis, Aeoden Sinclair",Joe Curtis,53.1%
2,XGB1,Sam Goin,"Joe Curtis, Brock Mantanona, Max McEnelly, Aeo...","Joe Curtis, Aeoden Sinclair",Joe Curtis,51.3%
3,XGB2,Sam Goin,"Rocco Welsh, Dylan Fishback, Max McEnelly, Aeo...","Rocco Welsh, Max McEnelly",Rocco Welsh,64.3%
4,VOTING,Sam Goin,"Rocco Welsh, Brock Mantanona, Max McEnelly, Ae...","Rocco Welsh, Aeoden Sinclair",Rocco Welsh,66.6%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs,149lbs,157lbs,165lbs,174lbs,184lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Rocco Welsh
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Joe Curtis
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Joe Curtis
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Brandon Cannon,Mitchell Mesenbrink,Levi Haines,Rocco Welsh
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Rocco Welsh



🔍 TESTING KEY 184lbs MATCHUPS

📊 Rocco Welsh vs Aeoden Sinclair
--------------------------------------------------
LOGREG  : Rocco Welsh (53.6%)
DT      : Aeoden Sinclair (53.1%)
XGB1    : Rocco Welsh (55.900001525878906%)
XGB2    : Rocco Welsh (79.5%)
VOTING  : Rocco Welsh (66.6%)
All Agree: ❌

📊 Max McEnelly vs James Conway
--------------------------------------------------
LOGREG  : Max McEnelly (72.4%)
DT      : Max McEnelly (74.6%)
XGB1    : Max McEnelly (81.0999984741211%)
XGB2    : Max McEnelly (79.0999984741211%)
VOTING  : Max McEnelly (75.4%)
All Agree: ✅

📊 Joe Curtis vs Silas Allred
--------------------------------------------------
LOGREG  : Joe Curtis (68.1%)
DT      : Joe Curtis (85.6%)
XGB1    : Joe Curtis (70.5%)
XGB2    : Joe Curtis (51.0%)
VOTING  : Joe Curtis (68.2%)
All Agree: ✅

📊 Jaden Bullock vs Dylan Fishback
--------------------------------------------------
LOGREG  : Dylan Fishback (58.2%)
DT      : Dylan Fishback (61.1%)
XGB1    : Dylan Fishback (64.0%)
XGB2

In [96]:
# ============================================
# CREATE 197lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 197lbs BRACKET")
print("="*100)

bracket_197 = Bracket(197, wrestlers_197)

# Pigtail match (33 vs 32)
bracket_197.add_pigtail('Karson Tompkins', 'Blake Schaffer')

# Round of 32 matchups
round_of_32_197 = [
    ('Josh Barr', 'PIGTAIL_WINNER'),            # (1) vs pigtail winner
    ('Dillon Bechtold', 'Branson John'),         # (17) vs (16)
    ('Angelo Posada', 'Brock Zurawski'),         # (9) vs (24)
    ('Evan Bates', 'Deanthony Parker'),          # (25) vs (8)
    ('Joey Novak', 'Kael Wisler'),               # (5) vs (28)
    ('Rune Lawrence', 'Luke Geog'),              # (21) vs (12)
    ('Bennett Berge', 'Wyatt Ingham'),           # (13) vs (20)
    ('Colton Hawks', 'Sonny Sasso'),             # (29) vs (4)
    ('Stephen Little', 'Kade Rule'),             # (3) vs (30)
    ('Zayne Lehman', 'Gabe Sollars'),            # (19) vs (14)
    ('Camden McDanel', 'Devin Wasley'),          # (11) vs (22)
    ('Gabe Arnold', 'Justin Rademacher'),        # (27) vs (6)
    ('Cody Merrill', 'Ben Vanadia'),             # (7) vs (26)
    ('Mikey Squires', 'Mac Stout'),              # (23) vs (10)
    ('Remy Cotton', 'Andrew Reall'),             # (15) vs (18)
    ('Kael Bennie', 'Rocky Elam')                # (31) vs (2)
]

bracket_197.add_round_of_32(round_of_32_197)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 197lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_197)}")
print(f"   Pigtail matches: {len(bracket_197.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_197.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_197.rounds['pigtail']) + len(bracket_197.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 197lbs matchups exist in predictions:")

key_matchups = [
    ('Josh Barr', 'Rocky Elam'),
    ('Sonny Sasso', 'Stephen Little'),
    ('Camden McDanel', 'Mac Stout'),
    ('Andrew Reall', 'Gabe Arnold'),
    ('Cody Merrill', 'Justin Rademacher')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 197lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 197lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_197 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_197[model] = bracket_197.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 197lbs
# ============================================

print("\n" + "="*100)
print("📊 197lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_197 = []
for model in models:
    res = results_197[model]
    summary_data_197.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_197_df = pd.DataFrame(summary_data_197)
display(summary_197_df)

# ============================================
# COMPARE ALL WEIGHTS SO FAR
# ============================================

print("\n" + "="*100)
print("📊 CROSS-WEIGHT CHAMPION COMPARISON")
print("="*100)

comparison_data = []
for model in models:
    champ_125 = results_125[model]['champion'] if 'results_125' in globals() else 'N/A'
    champ_133 = results_133[model]['champion'] if 'results_133' in globals() else 'N/A'
    champ_141 = results_141[model]['champion'] if 'results_141' in globals() else 'N/A'
    champ_149 = results_149[model]['champion'] if 'results_149' in globals() else 'N/A'
    champ_157 = results_157[model]['champion'] if 'results_157' in globals() else 'N/A'
    champ_165 = results_165[model]['champion'] if 'results_165' in globals() else 'N/A'
    champ_174 = results_174[model]['champion'] if 'results_174' in globals() else 'N/A'
    champ_184 = results_184[model]['champion'] if 'results_184' in globals() else 'N/A'
    champ_197 = results_197[model]['champion']
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': champ_125,
        '133lbs': champ_133,
        '141lbs': champ_141,
        '149lbs': champ_149,
        '157lbs': champ_157,
        '165lbs': champ_165,
        '174lbs': champ_174,
        '184lbs': champ_184,
        '197lbs': champ_197
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 197lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 197lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Josh Barr', 'Rocky Elam'),
    ('Sonny Sasso', 'Stephen Little'),
    ('Camden McDanel', 'Mac Stout'),
    ('Andrew Reall', 'Gabe Arnold'),
    ('Cody Merrill', 'Justin Rademacher')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 197)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 197lbs BRACKET

📋 197lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 197lbs matchups exist in predictions:
✅ Josh Barr vs Rocky Elam: FOUND
✅ Sonny Sasso vs Stephen Little: FOUND
✅ Camden McDanel vs Mac Stout: FOUND
✅ Andrew Reall vs Gabe Arnold: FOUND
✅ Cody Merrill vs Justin Rademacher: FOUND

🏆 SIMULATING 197lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 197lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Karson Tompkins vs Blake Schaffer: Blake Schaffer wins (51.5%)

🔹 ROUND OF 32
----------------------------------------
Josh Barr vs Blake Schaffer: Josh Barr wins (64.4%)
Dillon Bechtold vs Branson John: Dillon Bechtold wins (51.7%)
Angelo Posada vs Brock Zurawski: Angelo Posada wins (50.2%)
Evan Bates vs Deanthony Parker: Deanthony Parker wins (51.6%)
Joey Novak vs Kael Wisler: Joey Novak wins (59.1%)

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Blake Schaffer,"Josh Barr, Joey Novak, Stephen Little, Rocky Elam","Josh Barr, Rocky Elam",Josh Barr,50.2%
1,DT,Blake Schaffer,"Josh Barr, Bennett Berge, Zayne Lehman, Andrew...","Josh Barr, Zayne Lehman",Josh Barr,61.1%
2,XGB1,Blake Schaffer,"Josh Barr, Joey Novak, Zayne Lehman, Andrew Reall","Josh Barr, Zayne Lehman",Josh Barr,65.6%
3,XGB2,Karson Tompkins,"Josh Barr, Sonny Sasso, Camden McDanel, Mac Stout","Josh Barr, Camden McDanel",Josh Barr,73.7%
4,VOTING,Blake Schaffer,"Josh Barr, Joey Novak, Zayne Lehman, Rocky Elam","Josh Barr, Rocky Elam",Josh Barr,64.0%



📊 CROSS-WEIGHT CHAMPION COMPARISON


,Model,125lbs,133lbs,141lbs,149lbs,157lbs,165lbs,174lbs,184lbs,197lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Rocco Welsh,Josh Barr
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Joe Curtis,Josh Barr
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Joe Curtis,Josh Barr
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Brandon Cannon,Mitchell Mesenbrink,Levi Haines,Rocco Welsh,Josh Barr
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Rocco Welsh,Josh Barr



🔍 TESTING KEY 197lbs MATCHUPS

📊 Josh Barr vs Rocky Elam
--------------------------------------------------
LOGREG  : Josh Barr (50.2%)
DT      : Josh Barr (61.1%)
XGB1    : Josh Barr (77.30000305175781%)
XGB2    : Josh Barr (80.5999984741211%)
VOTING  : Josh Barr (64.0%)
All Agree: ✅

📊 Sonny Sasso vs Stephen Little
--------------------------------------------------
LOGREG  : Stephen Little (58.0%)
DT      : Stephen Little (61.1%)
XGB1    : Stephen Little (60.599998474121094%)
XGB2    : Sonny Sasso (77.0%)
VOTING  : Stephen Little (59.5%)
All Agree: ❌

📊 Camden McDanel vs Mac Stout
--------------------------------------------------
LOGREG  : Mac Stout (50.8%)
DT      : Mac Stout (61.1%)
XGB1    : Mac Stout (64.4000015258789%)
XGB2    : Camden McDanel (59.400001525878906%)
VOTING  : Mac Stout (56.0%)
All Agree: ❌

📊 Andrew Reall vs Gabe Arnold
--------------------------------------------------
LOGREG  : Andrew Reall (64.6%)
DT      : Andrew Reall (70.0%)
XGB1    : Andrew Reall (79.5%)

In [97]:
# ============================================
# CREATE 285lbs BRACKET
# ============================================

print("\n" + "="*100)
print("🏆 SETTING UP 285lbs BRACKET")
print("="*100)

bracket_285 = Bracket(285, wrestlers_285)

# Pigtail match (33 vs 32)
bracket_285.add_pigtail('Mason Rebuck', 'Emmanuel Ulrich')

# Round of 32 matchups
round_of_32_285 = [
    ('Yonger Bastida', 'PIGTAIL_WINNER'),       # (1) vs pigtail winner
    ('Vincent Mueller', 'Jim Mullen'),           # (17) vs (16)
    ('Cole Mirasola', 'Connor Barket'),          # (9) vs (24)
    ('Alex Semenenko', 'Ben Kueter'),            # (25) vs (8)
    ('Nick Feldman', 'Jarrett Stoner'),          # (5) vs (28)
    ('Juan Mora', 'Braxton Amos'),               # (21) vs (12)
    ('Spencer Lanogsa', 'Dayton Pitzer'),        # (13) vs (20)
    ('Luke Rasmussen', 'AJ Ferrari'),            # (29) vs (4)
    ('Taye Ghadiali', 'Jack Forbes'),            # (3) vs (30)
    ('Nate Schon', 'Koy Hopke'),                 # (19) vs (14)
    ('Devon Dawson', 'Trevor Tinker'),           # (11) vs (22)
    ('Hunter Catka', 'Nathan Taylor'),           # (27) vs (6)
    ('Konner Doucet', 'Luke Luffman'),           # (7) vs (26)
    ('Stephan Monchery', 'David Szuba'),         # (23) vs (10)
    ('Brady Colbert', 'Christian Carroll'),      # (15) vs (18)
    ('Brenan Morgan', 'Isaac Trumble')           # (31) vs (2)
]

bracket_285.add_round_of_32(round_of_32_285)

# ============================================
# VERIFY BRACKET SETUP
# ============================================

print(f"\n📋 285lbs Bracket Setup:")
print(f"   Total wrestlers: {len(wrestlers_285)}")
print(f"   Pigtail matches: {len(bracket_285.rounds['pigtail'])}")
print(f"   Round of 32 matches: {len(bracket_285.rounds['round_of_32'])}")
print(f"   Total first round matches: {len(bracket_285.rounds['pigtail']) + len(bracket_285.rounds['round_of_32'])}")
print(f"   Expected: 16 matches (1 pigtail + 15 round of 32)")

# ============================================
# VERIFY KEY MATCHUPS EXIST IN PREDICTIONS
# ============================================

print("\n🔍 Verifying key 285lbs matchups exist in predictions:")

key_matchups = [
    ('Yonger Bastida', 'Isaac Trumble'),
    ('AJ Ferrari', 'Taye Ghadiali'),
    ('Nick Feldman', 'Nathan Taylor'),
    ('Konner Doucet', 'Luke Luffman'),
    ('Ben Kueter', 'Koy Hopke')
]

for w1, w2 in key_matchups:
    matchup = master_pred_df[
        ((master_pred_df['home_wrestler'] == w1) & (master_pred_df['away_wrestler'] == w2)) |
        ((master_pred_df['home_wrestler'] == w2) & (master_pred_df['away_wrestler'] == w1))
    ]
    if len(matchup) > 0:
        print(f"✅ {w1} vs {w2}: FOUND")
    else:
        print(f"❌ {w1} vs {w2}: MISSING")

# ============================================
# SIMULATE 285lbs FOR ALL MODELS
# ============================================

print("\n" + "="*100)
print("🏆 SIMULATING 285lbs BRACKET")
print("="*100)

models = ['logreg', 'dt', 'xgb1', 'xgb2', 'voting']
results_285 = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"📊 {model.upper()} MODEL SIMULATION")
    print(f"{'='*60}")

    results_285[model] = bracket_285.simulate(master_pred_df, model, verbose=True)

# ============================================
# SUMMARY TABLE FOR 285lbs
# ============================================

print("\n" + "="*100)
print("📊 285lbs BRACKET SIMULATION SUMMARY")
print("="*100)

summary_data_285 = []
for model in models:
    res = results_285[model]
    summary_data_285.append({
        'Model': model.upper(),
        'Pigtail Winner': res['pigtail_winner'],
        'Quarterfinalists': ', '.join(res['quarterfinal_winners']),
        'Semifinalists': ', '.join(res['semifinal_winners']),
        'Champion': res['champion'],
        'Confidence': f"{res['champion_prob']*100:.1f}%"
    })

summary_285_df = pd.DataFrame(summary_data_285)
display(summary_285_df)

# ============================================
# COMPLETE CROSS-WEIGHT CHAMPION COMPARISON
# ============================================

print("\n" + "="*100)
print("📊 COMPLETE CROSS-WEIGHT CHAMPION COMPARISON - ALL 10 WEIGHTS")
print("="*100)

comparison_data = []
for model in models:
    comparison_data.append({
        'Model': model.upper(),
        '125lbs': results_125[model]['champion'] if 'results_125' in globals() else 'N/A',
        '133lbs': results_133[model]['champion'] if 'results_133' in globals() else 'N/A',
        '141lbs': results_141[model]['champion'] if 'results_141' in globals() else 'N/A',
        '149lbs': results_149[model]['champion'] if 'results_149' in globals() else 'N/A',
        '157lbs': results_157[model]['champion'] if 'results_157' in globals() else 'N/A',
        '165lbs': results_165[model]['champion'] if 'results_165' in globals() else 'N/A',
        '174lbs': results_174[model]['champion'] if 'results_174' in globals() else 'N/A',
        '184lbs': results_184[model]['champion'] if 'results_184' in globals() else 'N/A',
        '197lbs': results_197[model]['champion'] if 'results_197' in globals() else 'N/A',
        '285lbs': results_285[model]['champion']
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# ============================================
# TEST SPECIFIC 285lbs MATCHUPS
# ============================================

print("\n" + "="*100)
print("🔍 TESTING KEY 285lbs MATCHUPS")
print("="*100)

test_matchups = [
    ('Yonger Bastida', 'Isaac Trumble'),
    ('AJ Ferrari', 'Taye Ghadiali'),
    ('Nick Feldman', 'Nathan Taylor'),
    ('Konner Doucet', 'Luke Luffman'),
    ('Ben Kueter', 'Koy Hopke')
]

for w1, w2 in test_matchups:
    result = head_to_head(w1, w2, master_pred_df, 285)
    if 'error' not in result:
        print(f"\n📊 {result['matchup']}")
        print("-" * 50)
        for model, pred in result['results'].items():
            print(f"{model.upper():8}: {pred['winner']} ({pred['probability']}%)")
        print(f"{'All Agree':8}: {'✅' if result['all_agree'] else '❌'}")


🏆 SETTING UP 285lbs BRACKET

📋 285lbs Bracket Setup:
   Total wrestlers: 33
   Pigtail matches: 1
   Round of 32 matches: 16
   Total first round matches: 17
   Expected: 16 matches (1 pigtail + 15 round of 32)

🔍 Verifying key 285lbs matchups exist in predictions:
✅ Yonger Bastida vs Isaac Trumble: FOUND
✅ AJ Ferrari vs Taye Ghadiali: FOUND
✅ Nick Feldman vs Nathan Taylor: FOUND
✅ Konner Doucet vs Luke Luffman: FOUND
✅ Ben Kueter vs Koy Hopke: FOUND

🏆 SIMULATING 285lbs BRACKET

📊 LOGREG MODEL SIMULATION

🏆 285lbs - LOGREG MODEL

🔹 PIGTAIL ROUND
----------------------------------------
Mason Rebuck vs Emmanuel Ulrich: Mason Rebuck wins (61.4%)

🔹 ROUND OF 32
----------------------------------------
Yonger Bastida vs Mason Rebuck: Yonger Bastida wins (63.5%)
Vincent Mueller vs Jim Mullen: Vincent Mueller wins (61.4%)
Cole Mirasola vs Connor Barket: Connor Barket wins (53.4%)
Alex Semenenko vs Ben Kueter: Alex Semenenko wins (67.8%)
Nick Feldman vs Jarrett Stoner: Nick Feldman wins (67

,Model,Pigtail Winner,Quarterfinalists,Semifinalists,Champion,Confidence
0,LOGREG,Mason Rebuck,"Yonger Bastida, AJ Ferrari, Devon Dawson, Isaa...","Yonger Bastida, Isaac Trumble",Yonger Bastida,50.2%
1,DT,Mason Rebuck,"Yonger Bastida, Juan Mora, Taye Ghadiali, Isaa...","Yonger Bastida, Isaac Trumble",Yonger Bastida,61.1%
2,XGB1,Mason Rebuck,"Yonger Bastida, AJ Ferrari, Taye Ghadiali, Isa...","Yonger Bastida, Isaac Trumble",Yonger Bastida,51.3%
3,XGB2,Mason Rebuck,"Yonger Bastida, Nick Feldman, Taye Ghadiali, K...","Yonger Bastida, Konner Doucet",Yonger Bastida,59.0%
4,VOTING,Mason Rebuck,"Yonger Bastida, AJ Ferrari, Taye Ghadiali, Isa...","Yonger Bastida, Isaac Trumble",Yonger Bastida,58.8%



📊 COMPLETE CROSS-WEIGHT CHAMPION COMPARISON - ALL 10 WEIGHTS


,Model,125lbs,133lbs,141lbs,149lbs,157lbs,165lbs,174lbs,184lbs,197lbs,285lbs
0,LOGREG,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Rocco Welsh,Josh Barr,Yonger Bastida
1,DT,Luke Lilledahl,Jax Forrest,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Joe Curtis,Josh Barr,Yonger Bastida
2,XGB1,Luke Lilledahl,Marcus Blaze,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Joe Curtis,Josh Barr,Yonger Bastida
3,XGB2,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Brandon Cannon,Mitchell Mesenbrink,Levi Haines,Rocco Welsh,Josh Barr,Yonger Bastida
4,VOTING,Luke Lilledahl,Lucas Byrd,Jesse Mendez,Shayne Van Ness,Meyer Shapiro,Mitchell Mesenbrink,Levi Haines,Rocco Welsh,Josh Barr,Yonger Bastida



🔍 TESTING KEY 285lbs MATCHUPS

📊 Yonger Bastida vs Isaac Trumble
--------------------------------------------------
LOGREG  : Yonger Bastida (50.2%)
DT      : Yonger Bastida (61.1%)
XGB1    : Yonger Bastida (51.29999923706055%)
XGB2    : Yonger Bastida (65.0999984741211%)
VOTING  : Yonger Bastida (58.8%)
All Agree: ✅

📊 AJ Ferrari vs Taye Ghadiali
--------------------------------------------------
LOGREG  : Taye Ghadiali (53.3%)
DT      : Taye Ghadiali (53.1%)
XGB1    : Taye Ghadiali (55.0%)
XGB2    : AJ Ferrari (56.599998474121094%)
VOTING  : Taye Ghadiali (53.2%)
All Agree: ❌

📊 Nick Feldman vs Nathan Taylor
--------------------------------------------------
LOGREG  : Nick Feldman (52.3%)
DT      : Nick Feldman (61.1%)
XGB1    : Nick Feldman (64.5999984741211%)
XGB2    : Nick Feldman (75.80000305175781%)
VOTING  : Nick Feldman (63.1%)
All Agree: ✅

📊 Konner Doucet vs Luke Luffman
--------------------------------------------------
LOGREG  : Konner Doucet (53.9%)
DT      : Konner Douc